## Figures for Mt. Kenya publication


## Figures created for publication
* [Fig. 1](#fig_ts_periods): inventory maps
* Fig.2 is created in other repository (https://gitlab.empa.ch/leob/flexpart_mkn/-/blob/master/figures_footprints.ipynb?ref_type=heads)
* [Fig. 3](#fig_ts): Time series since 2020
* [Fig. 4](#fig_diurnal): diurnal cycles
* [Fig. 5](#fig_emission_contr): simulated and observed time series
* [Fig. 6](#fig_emission_scatter): scatter plots of simulated vs. observed time series
* [Fig. 7](#fig_emission_conribution_ts): time series of simulated emission contributions by sector
* [Fig. 8](#fig_emission_conribution_bars): bar plots of emission contributions by sector and seasons
* [Fig. A1](#fig_meteo): Meteo figure
* Fig A2 was created by Benjamin Brem
* [Fig. A3](#fig_ts_periods): Time series since 2002

In [ ]:
%matplotlib widget 
%load_ext autoreload 
%config InlineBackend.print_figure_kwargs = {'bbox_inches': None} # deactivate default whitespace removal in notebook

In [ ]:
# import 
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
import os
import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys
from pathlib import Path
import datetime as dt
from datetime import timedelta, datetime
import string
from cycler import cycler
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import itertools
import matplotlib as mpl
import mplotutils as mpu
import matplotlib.dates as mdates
import matplotlib.lines as mlines
import matplotlib.colors as mc
import polars as pl
from labellines import labelLine, labelLines
import glob
import cmcrameri.cm as cmc
import re
import scienceplots
import seaborn as sns
import scipy
from scipy.stats import linregress


from cartopy.feature import ShapelyFeature
from cartopy.io.shapereader import Reader,natural_earth
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import mplotutils as mpu
from matplotlib.colors import LogNorm, SymLogNorm
from matplotlib.ticker import LogFormatter, FuncFormatter
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
from suntime import Sun, SunTimeException


import emiproc
from emiproc.inventories.edgar import download_edgar_files, EDGARv8
from emiproc.grids import RegularGrid
from emiproc.regrid import remap_inventory
from emiproc.plots import plot_inventory
from emiproc.inventories.wetcharts import WetCHARTs
from emiproc.exports.hourly import export_hourly_emissions
from emiproc.inventories.utils import group_categories
from emiproc.exports.rasters import export_raster_netcdf
from emiproc.utilities import DAY_PER_YR

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)

from utils.utilities import find_best_grid_point, get_station_coords,form_xdate, get_anomalies, adjust_lightness, to_datetime
from utils import process_data
from utils import run_curve_fit
import tol_colors # color schemes from https://sronpersonalpages.nl/~pault
from utils.ccg_filter import ccg_filter as ccgfilt
from utils.ccg_filter import ccg_dates

from figure_props import FigureAspects, update_all_rcParams, adjust_lightness, cm2inch, save_fig_extact_size, custom_month_tick_formatter


from input import read_aerosols



### Figure properties

In [ ]:
is_poster = False # fontsize and figure size for posters

# figure style
plt.style.use(['science','nature']) # use the nice science style (+ nature for sans serif font) from SciencePlots package, but adapt it

############# Fonts and colors ################
## Colors
# high contrast color scheme, colorblind safe, grey-scale save, from  https://personal.sron.nl/~pault/
col_contr = tol_colors.tol_cset('high-contrast') #blue, yellow, red, black
col_contr_medium = tol_colors.tol_cset('medium-contrast')
col_bright = tol_colors.tol_cset('bright')
col_vibrant = tol_colors.tol_cset('vibrant') # original order: orange blue cyan magenta red teal grey
col_vibrant = [col_vibrant[i] for i in [1,0,5,3,2,4,6]] # use blue first
# access a specific color: e.g. getattr(col_contr_medium,'light_blue')

## colormap
cmap = tol_colors.tol_cmap('BuRd_discrete')

alphabets = string.ascii_lowercase

## Figure style:
# fontsize
fs=8 #caption fontsize in acp is 8.97

# for papers
figW=8.3

figW2 = 12 # in cm

# # textwidth in acp paper: 17.7cm
figW3 = 17.7


# for posters
if is_poster:
    figW=8.3
    figW2 = 25
    figW3 = 30

    fs = 16

figH = figW / FigureAspects.GOLDENRATIO.value
figH2 = figW2 / FigureAspects.GOLDENRATIO.value
figH3 = figW3 / FigureAspects.GOLDENRATIO.value

fs_cm = fs*0.0352778
###################################################



In [ ]:

## -------------------------TO ADAPT------------------------------##
# where to save figures:
if is_poster:
    dir_save = '../output/poster_figures202504/' 
else: 
    dir_save = '../output/publication_figures/' 
save_fig = True

# remove outliers or not
remove_outliers = False #remove outliers based ont uncertainty and statistics
remove_fire_events = True # remove spcific fire events (defined below)


# figure type
fig_format = 'pdf'
fig_dpi = 300

## Define analyses period
time1 = "2020-01-01"
time2 = "2024-12-31"


# define color of all ink (axes, labels, titles...). 
# Usually black, but could also be grey.
axes_col = 'black' #'dimgrey'
tit_col = 'black'#'black' # label and title colors
my_cols = col_contr #col_vibrant #col_contr
transp=False #save figure transparent or not
lgd_col = True # define if legend text should be colored or not
sig_transp = 0.1 #transparency of sigma shading
marker_transp = 1 # transparancy of markers when showing data behind smoothed lines


if is_poster:
    marker_size = 6 # size of markers
    ax_lw = .8 #linewidth of the axes (and ticks and zero line)
    pl_lw = 1.5 #linewidth of plotted lines
    lw_thick = 1.8 #+ 0.2 # thicker lines 
    lw_thin=1 # thinner lines
else:
    marker_size = 3 # size of markers
    ax_lw = .8 #linewidth of the axes (and ticks and zero line)
    pl_lw = 1 #linewidth of plotted lines
    lw_thick = 1.2 #+ 0.2 # thicker lines 
    lw_thin=0.4 # thinner lines
## -------------------------END ADAPT------------------------------##

# update RC parameters for figure layout:
update_all_rcParams(fs, pl_lw, ax_lw, lw_thin, lw_thick,marker_size, my_cols, axes_col, tit_col)

In [ ]:
# Define seasons:
my_seasons = {
    "JF": [1, 2],  # intermediate season
    "MAM": [3, 4, 5],  # long rains
    "JJAS": [6, 7, 8, 9],  # dry season
    #'S': [9], #not sure where September fits
    "OND": [10, 11, 12],  # short rains
}

seas_cols = {
    "JF": col_bright.purple,  # 1,2
    "MAM": col_bright.blue,  # 3,4,5
    "JJAS": col_bright.red,  # 6,7,8,9
    #'S': 'tab:grey',
    "OND": col_bright.green,  # 10,11,12
}

### Read all GHG data

In [ ]:
## Read all data
from input.read_wdc_data import AvailableData, create_data_reader

# File path
data_path = "../../data/"
external_data_path = "/project/leob/GAW/Kenya/data/" # on ddm
#external_data_path = r"C:\Users\leob\Documents\Data_analyses\Data"

# if New data is added to ./data folder, adapt the dictionary in AvailableData
all_data = list(AvailableData)
print(all_data)

#####---------- TO ADAPT ---------------#####
selected_data = ['CO2', 'CO2_flask', 
                 'CO', 'CO_flask', 
                 'CH4', 'CH4_flask', 
                 'O3'
                 ] # define data to read in. If empty, all data is used 
## 
processing_kwargs = { 
    'FLASK_FLAG_CORR' : True # exclude flagged flask-data
}
#####-----------------------------------#####

datasets = [] # initialize list of all datasets 
# read in data
for sel in (selected_data if selected_data else all_data):
    #define where the data has to be read from
    data_reader =  create_data_reader(data_path=data_path,dataset=sel,**processing_kwargs) #creates an instance of the desired data_reader class
    print(f"Data from {data_reader.__class__.__name__} for {sel}:")

    # call the data-reading function on that instance: 
    data = data_reader.read_data() 
    # call the data-processing
    data = data_reader.process_data(data)

    # prepare merged dataset
    data = data.drop(columns='endtime') # problem when merging datasets (because of NaT?), so better remove endtime
    ds = data.to_xarray()
    ds = ds.assign_coords(dataset=sel)
    ds['species'] = data_reader.species
    ds['unit']  = np.unique(ds.unit.dropna(dim='time'))[0]
    datasets.append(ds)

# save all in one xarray dataset
ds_all = xr.concat(datasets,dim="dataset")
ds_all

#### Remove outliers

We identified 2 fire events with local fires, that have a strong effect on the measurements: 
- March 2021 (2021-03-20 to 2021-03-28)
- Marcdh 2022 (2022-03-13 to 2022-03-27)

To activate that, set `remove_fire_events` above to True

In [ ]:
# Define events with local fire activity based on GFAS data
fire_dates_Mar2022 = slice('2022-03-13','2022-03-27')
fire_times_Mar2022 = ds_all.sel(time=fire_dates_Mar2022).time.values

fire_dates_Mar2021 = slice('2021-03-20','2021-03-28')
fire_times_Mar2021 = ds_all.sel(time=fire_dates_Mar2021).time.values

fire_dates_JA2024 = slice('2024-09-18','2024-09-25')
fire_times_JA2024 = ds_all.sel(time=fire_dates_JA2024).time.values

if remove_fire_events:    # remove fire events from the dataset. Use a workaround, because xarray adds time dimension to all variables
    ds_withtime = ds_all.drop([ var for var in ds_all.variables if not 'time' in ds_all[var].dims ])
    ds_timeless = ds_all.drop([ var for var in ds_all.variables if     'time' in ds_all[var].dims ])
    ds_withtime = ds_withtime.where(ds_withtime.time.isin(fire_times_Mar2022)==False) #don't use the fire event times 2022
    ds_withtime = ds_withtime.where(ds_withtime.time.isin(fire_times_Mar2021)==False) #don't use the fire event times 2021
    ds_withtime = ds_withtime.where(ds_withtime.time.isin(fire_times_JA2024)==False) #don't use the fire event times 2024
    ds_all_rem_out = xr.merge([ds_timeless, ds_withtime])

if remove_outliers:
    # Remove outliers that exceed 10*stdedeviation_mean and 4* the zscore 
    # remove outliers for each species sperarately
    ds_all_rem_out = ds_all.copy(deep=True) #use deepcopy, otherwise it replaces values in ds_all!
    outlier_masks = {}
    for ds in np.unique(ds_all.species):
        print(f"remove outliers for {ds}:")
        ds_all_rem_out.loc[dict(dataset=ds)], mask_removed_outliers= process_data.rem_out(ds_all.sel(dataset=ds), std_fac=10, z_threshold=4)
        outlier_masks[ds] = mask_removed_outliers

    #ds_all = ds_all_rem_out # use the removed outlier data!


### Read in aerosol data and remove fire event

In [ ]:
# Read in aerosol data and extract Black carbon
#aerosol_data_dir = "../../data/level2/L2_AEROSOL_data_bachelorthesis Mike Baumann"
#ds_ae, ds_neph = read_aerosols.aerosol_data_to_dataset(aerosol_data_dir)

# make a full dataset with wavelength dependence and calculated aerosol properties (SAE, SSA, AAE)
#ds_aerosols = read_aerosols.aerosols_to_full_dataset(ds_ae, ds_neph) 

# Finally, I dont need that full aerosol data for the publication
# Just read in the prepared black carbon data (for AE31 and AE33):
ds_aerosols = read_aerosols.read_equivalent_black_carbon("../../data/level2/mkn/")
ds_aerosols = ds_aerosols.sel(time=slice(time1, time2))

## Remove fire events/outliers from the aerosol data

if remove_fire_events:
    # remove fire events from the dataset. Use a workaround, because xarray adds time dimension to all variables
    ds_withtime = ds_aerosols.drop([ var for var in ds_aerosols.variables if not 'time' in ds_aerosols[var].dims ])
    ds_timeless = ds_aerosols.drop([ var for var in ds_aerosols.variables if     'time' in ds_aerosols[var].dims ])
    ds_withtime = ds_withtime.where(ds_withtime.time.isin(fire_times_Mar2022)==False) #don't use the fire event times 2022
    ds_withtime = ds_withtime.where(ds_withtime.time.isin(fire_times_Mar2021)==False) #don't use the fire event times 2021
    ds_withtime = ds_withtime.where(ds_withtime.time.isin(fire_times_JA2024)==False) #don't use the fire event times 2024
    ds_aerosols_rem_out = xr.merge([ds_timeless, ds_withtime])

if remove_outliers:
    outlier_masks = {}
    for var_name in np.unique(list(ds_aerosols.keys())[0:4]):
        print(f"remove outliers for {var_name}:")
        drop_vars = list(ds_aerosols.keys())
        drop_vars.remove(var_name)
        ds_aer_removed_outliers, mask_aer_no_outliers = process_data.rem_out(ds_aerosols_rem_out.drop(drop_vars),vars=[var_name], use_unc=False, std_fac=10, z_threshold=6,plot_timeseries=False)
        ds_aerosols_rem_out[var_name] = ds_aer_removed_outliers[var_name]
        outlier_masks[var_name] = mask_aer_no_outliers

      


#### Read in o3sonde data

In [ ]:
# read in the o3sonde data that Jörg selected for MKN-pressure levels
o3sonde = pl.read_parquet('../../data/level2/nrb/ecc/ecc_ozone_nrb_vs_in_situ_ozone_mkn.parquet')


In [ ]:
df_o3sonde = o3sonde.to_pandas() # note that the dataframe is a polars dataframe (check: print(type(o3sonde['dtm']))! need to convert to pandas for some operations
# Create a pandas DataFrame with 'dtm' as the index
#time_df = pd.DataFrame({'dtm': time})
df_o3sonde.set_index('dtm', inplace=True)
# take the average for values with the same timestamp
df_o3sonde = df_o3sonde.groupby('dtm').mean()
# transform to dataset and rename the time dimension
ds_o3sonde = df_o3sonde.to_xarray()
ds_o3sonde = ds_o3sonde.rename({'dtm':'time'})

# Resample the data to daily frequency, but skip days with missing data
ds_o3sonde_daily = ds_o3sonde.resample(time='1D').mean()

plev = 700 #660  # o3sonde pressure level that was preselected for MKN (620,660,700mb)
o3sonde_daily = (ds_o3sonde_daily.where(ds_o3sonde_daily[f'o3_ecc_{plev}'] > 0, drop=True)[f'o3_ecc_{plev}']).sel(time=slice(time1, time2)) #in ppb

#### Read in meteo data

In [ ]:
from input import get_meteo
ds_meteo_new = get_meteo.get_meteo_timeseries_new(data_path=data_path,yr_start=2022) #starting in 2022
#ds_meteo_old = get_meteo.get_meteo_timeseries_old(data_path=data_path) #until 2022 -> not needed anymore, 2022 included in new data
# attention, the data is not processed! seem to have some duplicates in dates?

# remove date duplicates
#ds_meteo_new = ds_meteo_new.drop_duplicates('time')
#ds_meteo_old = ds_meteo_old.drop_duplicates('time')
# merge the two datasets
#ds_meteo = xr.concat([ds_meteo_old, ds_meteo_new], dim='time')

ds_meteo = ds_meteo_new.sortby('time')

# plt.figure()
# ds_meteo_old.temperature.plot()
# ds_meteo_new.temperature.plot()
# plt.show()

In [ ]:
# restrict meteo data to 2022 to 2024 (no data before?)
meteo_t1 = "2022-01-01"
meteo_t2 = "2024-12-31"

# Daily means
ds_daily = ds_meteo.sel(time=slice(meteo_t1, meteo_t2)).resample(time="1D")
valid_counts = ds_daily.count()#.temperature

# meteo data
# exlude days where we have less than 18 valid hourly measurements
valid_meteo_per_day = 18
temp_D = (
    ds_daily.mean().where(valid_counts["temperature"] >= valid_meteo_per_day)["temperature"]
)
precip_D = (
    ds_daily.sum().where(valid_counts["precipitation"] >= valid_meteo_per_day)["precipitation"]
)



### Meteo figure <a class="anchor" id="fig_meteo"></a>

In [ ]:
# just choose the nearest MKN grid



col_bright = tol_colors.tol_cset("bright")
## 1 Figure with all variables, time series and seasonal cycle

figW_temp = figW
figH_temp = figH2

# Time series
fig, axs = plt.subplots(
    2,
    1,
    figsize=(cm2inch((figW_temp,figH_temp)))
)

# Time series
ax1 = axs[0]


## First plot: temperature and precipitation daily time series
temp_col = col_bright.red
temp_D.plot(
    ax=ax1, label="Temperature", c=temp_col, lw=0.8
)  # .resample(time='1D').mean()

ax1.set_title("")
# ax1.legend(loc='upper left', fontsize=8)
ax1.set_xlabel("")

# color axis
ax1.spines["left"].set_color(temp_col)
ax1.tick_params(axis="y", colors=temp_col)
ax1.tick_params(axis="y", which="minor",colors=temp_col)
ax1.set_ylabel("Daily temperature (°C)")
ax1.yaxis.label.set_color(temp_col)


ax12 = ax1.twinx()
precip_color = col_bright.blue  # (0, 0, 1, 0.6)
# precip_D.plot(ax=ax12, label='Precipitation',c='k',alpha=0.6,lw=0.8) # daily precipitation as lines
# precipitation as bars:
ax12.bar(
    precip_D["time"].values,
    precip_D.values,
    color=precip_color,
    # alpha=0.6,
    width=1.0,  # adjust bar width (in days)
    label="Precipitation",
)

ax12.set_title("")
ax12.set_ylabel("Daily precip. (mm)")
#ax12.set_ylim(0, 200)


# color right axis
ax12.spines["left"].set_visible(False) # to avoid overwriting left axis
ax12.spines["right"].set_visible(True)
ax12.spines["right"].set_color(precip_color)
ax12.yaxis.label.set_color(precip_color)
ax12.tick_params(axis="y", colors=precip_color)

ax12.xaxis.set_major_locator(mdates.YearLocator(1))
ax12.xaxis.set_minor_locator(mdates.MonthLocator())
ax12.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))


## seasonal cycles /monthly data
months = range(1, 13)
# temp_M = ds_meteo['temperature'].sel(time=slice(meteo_t1,meteo_t2)).resample(time='MS').mean()
# t_meteo_grouped = temp_M.groupby('time.month').mean(dim='time') # not needed to take monthly means first for T
temp_grouped = (
    ds_meteo["temperature"]
    .sel(time=slice(meteo_t1, meteo_t2))
    .groupby("time.month")
    .mean(dim="time")
)

# precip
precip_M = (
    ds_meteo["precipitation"]
    .sel(time=slice(meteo_t1, meteo_t2))
    .resample(time="MS")
    .sum(dim="time")
)
precip_m_grouped = precip_M.groupby("time.month").mean(
    dim="time"
)  # take the mean of monthly precipitation sums over the years


ax2 = axs[1]
ax2.plot(months, temp_grouped, color=temp_col, label="T", zorder=2)
# sns.lineplot(data = ds_sel.groupby('time.month').mean(dim='time')['tmin'].to_dataframe(),x='month',y='tmin', ax=axs[1], color='red')
ax2.set_title("")
ax2.legend(loc="upper left")
ax2.set_xlabel("Month")

# color axis
ax2.spines["left"].set_color(temp_col)
ax2.set_ylabel("Temperature (°C)")
ax2.yaxis.label.set_color(temp_col)
ax2.tick_params(axis="y", colors=temp_col)
ax2.tick_params(axis="y", which="minor",colors=temp_col) #color minor y-ticks
ax2.tick_params(axis="x", which="minor", bottom=False) #remove minor x-ticks
#same ylims as for ax1:
ax2.set_ylim(ax1.get_ylim())

ax22 = ax2.twinx()
ax22.bar(
    months, precip_m_grouped, color=precip_color, label="Precipitation", zorder=1
)  # , alpha=0.6
# sns.barplot(data = precip_m_grouped.to_dataframe(),x='month',y='precip', ax=ax22)
ax22.set_title("")
ax22.legend(ncols=2, loc="upper right")

# color right axis
ax22.spines["right"].set_visible(True)
ax22.spines["right"].set_color(precip_color)
ax22.yaxis.label.set_color(precip_color)
ax22.tick_params(axis="y", colors=precip_color)
ax22.set_ylabel("Monthly precip. (mm)", color=precip_color)

ax22.set_xticks(
    months,
    [dt.datetime.strptime(str(month), "%m").strftime("%b") for month in months],

)
ax2.set_zorder(ax22.get_zorder() + 1)  # bring line axis above bars
ax2.patch.set_visible(False)  # make its background transparent

ax1.set_title(f"Temperature and precipitation at Mt. Kenya", loc='left')

# align ylabels
fig.align_ylabels(axs)

if save_fig:
    file_name_fig = f"{dir_save}MKN_meteo.{fig_format}"
    #file_name_fig = f"/newhome/leob/Documents/git-repos/gawkenya/analyses/output/publication_figures/MKN_meteo.{fig_format}"
    ## correct figure size when saving
    save_fig_extact_size(
        fig,
        file_name_fig,
        figsize_cm = (figW_temp, figH_temp),  # Width & height in cm
        margins_cm=(1.5, 1.3, 0.4, 1.3),  # (left, bottom, right, top) in cm
        fig_dpi=fig_dpi,
        fig_format=fig_format)

In [ ]:
# print min and max temperature values and the corresponding date
temp_sel = ds_meteo["temperature"].sel(time=slice(meteo_t1, meteo_t2))
min_value = temp_sel.min().item()
min_date = temp_sel.time[temp_sel.argmin().item()].values
print(f"Minimum temperature: {min_value}, Date: {min_date}")

# same for max
max_value = temp_sel.max().item()
max_date = temp_sel.time[temp_sel.argmax().item()].values
print(f"Maximum temperature: {max_value}, Date: {max_date}")

#daily min and max temperature
temp_min = (
    ds_meteo["temperature"]
    .sel(time=slice(meteo_t1, meteo_t2))
    .resample(time="1D")
    .min()
)
temp_max = (
    ds_meteo["temperature"]
    .sel(time=slice(meteo_t1, meteo_t2))
    .resample(time="1D")
    .max()
)


# print mean values of daily tmin and tmax
print(f"Mean daily min temperature: {temp_min.mean().values}")
print(f"Mean daily max temperature: {temp_max.mean().values}")



plt.figure()
temp_min.plot(label='Daily min temperature')
temp_max.plot(label='Daily max temperature')
plt.legend()
plt.show()


## GHG time series plots

This figure shows all the time series of the 4 GHG and Black carbon
In addition to the hourly measurements, I plot 3h CAMS data (6h for methane)

Some specifications: 
- The seasonal cycles are showing the averaged values for each day of the year, using the smoothed line fit
- We identified 2 local fire events, they are included here in the data (therefore also in the seasonal cycles)
    - when they are removed (ds_all_rem_out instead of ds_all and ds_aerosols_rem_out instead of ds_aerosols), the sesonal cycle look slightly different (especially for BC anc CO in march)

In [ ]:
# Define fit paramaters for the curve fitting

# Default values
fit_params_defaults = {'shortterm': 80, #Short term cutoff value in days for smoothing of data
                'longterm': 667, # smoothing in days. Default: 667
                'numpolyterms': 3, # use only 2 for less than 3 years of data, otherwise use 3 (=quadratic fit)
                'sampleinterval': 1 / 24,  # 1h
                'numharmonics': 4}

In [ ]:
## Helper functions for the figure


# function to plot each subplot
# time series plot
def plot_data(
    ds_temp,
    ds_temp_fit,
    label_meas="",
    label_fit="fit",
    ax=None,
    detrend=False,
    plot_measurements=True,
    plot_fit=True,
    **kwargs,
):
    """
    Plot the data and the fit
    detrend: if True, plot the detrended data
    """
    if ax is None:
        ax = plt.gca()

    ## get initial figure properties
    initial_color = kwargs["color"]
    initial_ls = kwargs["ls"]
    initial_zorder = kwargs["zorder"]

    if plot_measurements:
        kwargs["ls"] = ""  # no line for measurements
        pl = ds_temp.plot(
            ax=ax,
            alpha=marker_transp,
            label=label_meas,  # label[0:4],
            # markeredgewidth=2,
            rasterized=True,
            **kwargs,
        )
    else:
        pl = []
    # plot fit
    if plot_fit:
        kwargs["color"] = adjust_lightness(
            initial_color, amount=1.5
        )  # adapt hue of initial color (lighter)
        kwargs["marker"] = ""  # no marker for fit
        kwargs["ls"] = initial_ls
        kwargs["zorder"] = initial_zorder + 1  # plot fit on top of measurements
        pl_fit = ds_temp_fit["smoothed_vals"].plot(
            ax=ax, label=label_fit, **kwargs)
        if detrend:
            # plot detrended fit
            kwargs["color"] = adjust_lightness(
                initial_color, amount=0.8
            )  # adapt hue of initial color (darker)
            kwargs["ls"] = ":"  # dotted line for detrended
            ax.plot(
                ds_temp_fit.time,
                ds_temp_fit["seasonal_detrend"] + ds_temp_fit["smoothed_vals"].mean(),
                label="detrended fit",
                **kwargs,
            )  ## Add mean value to detrended to obtain same magnitude
    else:
        pl_fit = []
    return pl, pl_fit


# seasonality plot
def plot_cycle(
    ds_temp,
    ds_temp_fit,
    s,
    freq="month",
    ax=None,
    with_trend=False,
    plot_smoothed=True,
    label="",
    **kwargs,
):
    """
    Plot seasonal cycle of the data
    with_trend:  if True, plot the seasonal cycle with trend (non-detrended)
    plot_smootehd:  Plot the smoothed or the original data
    """

    ## get a different hue of the color used
    # Decrease the hue value
    initial_color = kwargs["color"]

    if ax is None:
        ax = plt.gca()
    ref = (
        ds_temp.mean()
    )  # use mean value from measurements to obtain positive/neg. seasonality (ds_temp-ref) or absolute values (ds_temp_fit +ref)

    if freq == "hour":
        kwargs["color"] = adjust_lightness(
            initial_color, amount=1.2
        )  # adapt hue of initial color (lighter)

        xvals = (ds_temp - ref).groupby(f"time.{freq}").mean()
        pl = xvals.plot(ax=ax, label=f"{label}", **kwargs)
        std = (
            (ds_temp - ref).groupby(f"time.{freq}").std()
        )  # standarddeviation of the grouped values of the measurements
        # add stdev
        ax.fill_between(
            xvals[freq],
            xvals,
            xvals + std,
            alpha=0.3,
            label=f"std. dev.",
            color=kwargs["color"],
        )
        ax.fill_between(
            xvals[freq],
            xvals,
            xvals - std,
            alpha=0.3,
            label=f"std. dev.",
            color=kwargs["color"],
        )
    else:
        # normal seasonal cycle
        if with_trend:
            kwargs["ls"] = ":"  # dotted line for not detrended
            kwargs["color"] = adjust_lightness(
                initial_color, amount=0.8
            )  # adapt hue of initial color (darker)
            # plot also the normal seasonal cycle (with trend) in addition
            (ds_temp - ref).groupby(f"time.{freq}").mean().plot(
                ax=ax, label=f"{label} not detrended", **kwargs
            )
        # detrended seasonal cycle
        kwargs["color"] = adjust_lightness(
            initial_color, amount=1.2
        )  # adapt hue of initial color (lighter)

        if plot_smoothed:
            xvals = ds_temp_fit["seasonal_detrend"].groupby(f"time.{freq}").mean()
            std = (
                ds_temp_fit["seasonal_detrend"].groupby(f"time.{freq}").std()
            )  # standarddeviation of the grouped values of the measurements
        else:
            xvals = (ds_temp - ref).groupby(f"time.{freq}").mean()
            std = (ds_temp - ref).groupby(f"time.{freq}").std()

        pl = xvals.plot(ax=ax, label=f"{label}", **kwargs)  # label=f"{label} detrended"

        # add stdev
        ax.fill_between(
            xvals[freq],
            xvals,
            xvals + std,
            alpha=0.3,
            label=f"std. dev.",
            color=kwargs["color"],
        )
        ax.fill_between(
            xvals[freq],
            xvals,
            xvals - std,
            alpha=0.3,
            label=f"std. dev.",
            color=kwargs["color"],
        )
    return pl

### Figure time series since 2002 (with different periods) <a class="anchor" id="fig_ts_periods"></a>

In [ ]:
# Select data to plot

# species to plot
my_species_full_period = [ 'CO2', 'CH4', 'O3', 'CO']

# only select data where the species is in my_species (including flask data)
ds_data = ds_all.where(ds_all.species.isin(my_species_full_period), drop=True)
ds_data_rem_out = ds_all_rem_out.where(ds_all_rem_out.species.isin(my_species_full_period), drop=True)


In [ ]:
# Define fit paramaters for the curve fitting

fit_properties = {}
for dataset in ds_data.dataset:
    dataset_name = dataset.values.item()  # Convert numpy array to a hashable type
    fit_properties[dataset_name] = fit_params_defaults.copy()
    
# Update fit_properties with non-default values if necessary:
# Remove sampleinterval for flask data:
for dataset in fit_properties:
    if '_flask' in dataset:
        fit_properties[dataset]['sampleinterval'] = 0 #if 0, determine from xp (time)

#fit_properties['CO']['numpolyterms'] = 3


In [ ]:
## Define the data periods I want to compare/plot 

t0 = ds_all.isel(time=0)
t1 = ds_all.isel(time=-1)
print(f"Total time period with data:: {t0.time.values} to {t1.time.values}")

compare_periods = {
    'A': (t0.time.values, dt.datetime(2006,12,31)), #old and flask data
    #'B': (dt.datetime(2008,1,1), dt.datetime(2011,12,31)), #only flask data
    'B': (dt.datetime(2008,1,1), dt.datetime(2019,12,31)), # flask data and older ozone data (before 2020)
    'C': (dt.datetime(2020,1,1), t1.time.values), # only new data
    #'C': (dt.datetime(2014,1,1), t1.time.values) # ozone starting in 2014 and new data (after 2020)
}


In [ ]:
# ## All GHG data since 2002 with seasonal cycles

##---- Plotting definitions ----##
figW_temp = figW3
figH_temp = figH3
# plot seasonal or daily cycles
plot_seas = True #if false, plot daily cycle instead of seasonal
seas_freq = 'month' # 'month' or 'dayofyear'
with_trend = False #if true, plot detrended data, otherwise detrend seasonal cycle but dont plot it as extra line in the time series plot

default_colors = ['C2', 'C1', 'C0', 'C3'] if len(compare_periods) >2 else ['C0', 'C1', 'C2', 'C3'] # make third period blue
#default_colors = ['C0', 'C0', 'C0', 'C3'] if len(compare_periods) >2 else ['C0', 'C1', 'C2', 'C3'] # make third period blue
color_iterator = itertools.cycle(default_colors)



##--- Start figure  ---##
# Make a mosaic grid, where each subplot is called either the species (e.g. CO2) or the species seasonal cycle (ew.g. CO2_seas)

mosaic_grids =  [[m, m+'_seas']  for m in my_species_full_period] # put [] around to start new row
gs_kw = dict(width_ratios=[3, 1]) #column and width ratios
subfig_height = 2.5 if len(my_species_full_period) == 1 else 2
fig, axd = plt.subplot_mosaic(mosaic_grids, 
                              gridspec_kw=gs_kw, figsize=(cm2inch((figW_temp,figH_temp))),
                              layout="constrained",
                              sharex=False, sharey=False)
                              
for period, (start_date, end_date) in compare_periods.items():
    print(f"Period {period}: {start_date} to {end_date}")
    # Filter the data for the current period
    period_data = ds_data.sel(time=slice(start_date, end_date))

    # Define figure properties for this period
    fig_properties = {
        'color': next(color_iterator),
        'marker': '.',
        'ls':'-'
    }
    
    # Write Period A, B and C just below the time axis in the color of the different periods
    # write it on top of the first axis
    # ax_sel = axd[my_species_full_period[0]] # first subplot
    # ax_sel.text(start_date, 400, f'Period {period}',color=fig_properties['color'], ha='left') #, transform=ax_sel.transAxes

    # ===================================
    ## Loop through all datasets and plot the data
    for i,dataset in enumerate(period_data.dataset):
        # Get the data for the current dataset and period
        data = period_data.sel(dataset=dataset)["value"]
        species = period_data.sel(dataset=dataset).species.values
        species_str = re.sub(r"(\d+)", r"$_\1$", str(species))
        #print(f"Dataset: {dataset.item()}, Species: {species}")

        if data.size > 0 and np.all(np.isnan(data))==False: #if we have data
        
            # Define fit properties for this dataset
            # Perform the curve fitting and obtain the interpolated time axis
            filt, df_interp, ds_interp = run_curve_fit.run_ccgfilter(
                ds=data,
                dataset_str=dataset.item(),
                t1=start_date,
                t2=end_date,
                **fit_properties[dataset.item()]
            )
            
            # ===================================
            # Plot the data and the fit
            # adapt figure properties for flask data
            if dataset.item() == 'CO_flask':
                # CO_flask is the only flask data that we have in parallel with continuous measurements
                # Therefore, we plot it with a different marker and linestyle
                fig_properties['marker'] = 'x'
                #fig_properties['color'] = 'lightgrey'
                fig_properties['zorder'] = 5
                fig_properties['ls'] = ':'
            elif '_flask' in dataset.item():
                # different marker for flask measurements
                fig_properties['marker'] = 'x'
                fig_properties['zorder'] = 5
                fig_properties['ls'] = ':'
            else:
                # normal properties
                fig_properties['marker'] = '.'
                fig_properties['ls'] = '-'            
                fig_properties['zorder'] = 5

           # ===================================
            ## Time series figure
            ax_ts = axd[species.item()]
            ## Seasonal cycle figure
            ax_seas = axd[species.item()+'_seas']


            legend_entry = f"curve fit {'flask' if '_flask' in dataset.item() else ''}({fit_properties[dataset.item()]['shortterm']} days)"
            plot_data(data,                       
                      ds_interp,
                      label_fit = legend_entry, 
                      ax= ax_ts,
                      detrend= with_trend
                      ,**fig_properties)

            # ===================================
            #plot seasonal cycle
            freq = seas_freq if plot_seas else 'hour' #for seasonal cycle, use 'dayofyear' (smoother line) or 'month', for daily cycle (if plot_seas=False) use 'hour'
            # no markers for seasonal cycle
            fig_properties['marker'] = '' #no marker for seasonal cycle
            pl_seas = plot_cycle(
                data,
                ds_interp,
                species,
                freq= freq,
                ax=ax_seas,
                with_trend=with_trend,
                label = period,
                **fig_properties
            )

            # ===================================
            #---- figure properties ----#

            # axes properties time series
            ax_ts.set_ylabel(f"{species_str} ({period_data.sel(dataset=dataset).unit.values})")
            #ax_ts.set_xlim(np.array([t1, t2], dtype="datetime64"))
            ax_ts.spines["top"].set_visible(False)
            ax_ts.spines["right"].set_visible(False)
            ax_ts.set_title("")
            # be sure to have same x-axis for all plots
            ax_ts.set_xlim([pd.to_datetime("2002-01-01"), pd.to_datetime(time2)])

            if species != my_species_full_period[-1]:  # all except lowest axis
                plt.setp(
                    ax_ts.get_xticklabels(), visible=False
                )  # remove xticklabels except for lowest plot
                ax_ts.set_xlabel("")        
            else: #lowest axis
                ax_ts.set_xlabel("Time")                
                ax_ts.xaxis.set_major_locator(mdates.YearLocator(2))
                ax_ts.xaxis.set_minor_locator(mdates.MonthLocator(1))
                ax_ts.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

            # axes properties seasonal cycle
            # ax_seas.set_yticklabels('') #this removes ylabels for both axes! Therefore better use:
            #ax_seas.tick_params(labelleft=False) #I need the labels if I use other units for the seasoal cycle!!
            ax_seas.set_title("")
            ax_seas.set_ylabel("")
            ax_seas.spines["top"].set_visible(False)
            ax_seas.spines["right"].set_visible(False)
            if plot_seas:
                if freq== 'month':
                    months = range(1, 13, 3) #major ticks every 3 months
                    ax_seas.set_xticks(
                        months,
                        [dt.datetime.strptime(str(month), "%m").strftime("%b") for month in months],
                    )
                    ax_seas.xaxis.set_minor_locator(ticker.FixedLocator(range(1, 13, 1))) # minor tick every month
                    #rotate the xlabels
                    #plt.setp(ax_seas.xaxis.get_majorticklabels(), rotation=45)
                elif freq == 'dayofyear':
                    ax_seas.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
                    ax_seas.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
                    ax_seas.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
                    #rotate the xlabels
                    plt.setp(ax_seas.xaxis.get_majorticklabels(), rotation=45)

            #ax_seas.grid(color='lightgrey',zorder=0,axis='x')
            ax_seas.set_axisbelow(False)

            # different for lowest plot
            if species != my_species_full_period[-1]:  # all except lowest axis
                plt.setp(
                    ax_seas.get_xticklabels(), visible=False
                )  # remove xticklabels except for lowest plot
                ax_seas.set_xlabel("")
                
# ===================================
## --- Final figure properties ---##
handles, labels = ax_ts.get_legend_handles_labels()
# manually adapt legend:
#labels[0] = "1h data"
ax0 = list(axd.values())[0] #first subplot (upper left)
if with_trend:
    ax0.legend(handles, labels, loc="lower right")
fig.align_ylabels()

# Add a manual legend showing the shape of the dots to the upper left subplot
marker_flask = mlines.Line2D([], [], color='k', marker='x', linestyle='None', label='Flask data')
marker_other = mlines.Line2D([], [], color='k', marker='.', linestyle='None',label='Continuous data')
line_flask = mlines.Line2D([], [], color='k', marker='', linestyle='-', label=f"Curve fit ({fit_properties[dataset.item()]['shortterm']} days)")
line_other = mlines.Line2D([], [], color='k', marker='', linestyle=':', label=f"Curve fit ({fit_properties[dataset.item()]['shortterm']} days) flask data")
ax0.legend(handles=[marker_other,marker_flask,line_flask,line_other], loc='upper center', ncols=2)

# Add the legend to the upper right subplot (cycles)
ax_sel = axd[my_species_full_period[0] +'_seas']
handles, labels = ax_sel.get_legend_handles_labels()
#labelLines(ax_sel.get_lines(), zorder=2.5) #labels on lines
line_other = mlines.Line2D([], [], color='k', marker='', linestyle='-', label=f"Continuous data")
line_flask = mlines.Line2D([], [], color='k', marker='', linestyle=':', label=f"Flask data")
ax_sel.legend(handles=[line_other,line_flask], loc='upper right', ncols=1)
#ax_sel.legend(handles, labels, loc='upper left')

# place title on first axis
ax0.set_title("Mt. Kenya measurements and curve fit", loc="left")
ax1 = list(axd.values())[1] #second subplot (upper right)
if plot_seas:
    tit_right = "Seasonal cycle (detrended)"
else:
    tit_right = 'Diurnal cycle'
ax1.set_title(tit_right, loc="center")

ax_seas.set_xlabel("Time")
ax_ts.set_xlabel("Months")



            
# subplot labelling (do that only once)
for ii,i_spec in enumerate(my_species_full_period):
    species_str = re.sub(r"(\d+)", r"$_\1$", str(i_spec))
    axis_ts = axd[i_spec]
    axis_seas = axd[i_spec+'_seas']
    for ax_sel, igap in zip([axis_ts,axis_seas],[0, len(my_species_full_period)]):
        ax_sel.text(
            0.01,
            1,
            (f"({alphabets[ii+igap]}) {species_str}"),
            transform=ax_sel.transAxes,
            bbox=dict(fc="white", ec="none", alpha=0),
            zorder=2,
            fontsize="small",
            verticalalignment="top"
        )

# # plot in each subplot a vertical line for the compare_periods (not working yet)
# for ax in axd.values():
#     if '_seas' in ax.get_label():
#         continue
#     else: 
#         for period, (start_date, end_date) in compare_periods.items():
#             ax.axvline(start_date, color='k', linestyle='--', alpha=0.5)
#             ax.axvline(end_date, color='k', linestyle='--', alpha=0.5)
#             # Create a rectangle patch with transparent background
#             #rect = plt.Rectangle((start_date, ax.get_ylim()[0]), end_date - start_date, ax.get_ylim()[1], facecolor=next(color_iterator), edgecolor='none')
#             # Add the rectangle patch to the plot
#             #ax.add_patch(rect)

str_seas = 'seas' if plot_seas else 'daily'
str_trend = '_detrended' if with_trend else ''
str_freq = '_' + seas_freq if plot_seas else ''

# if not all species are plotted: adapt name
if len(my_species_full_period) <4:
    str_spec = '_' + ''.join(my_species_full_period)
else:
    str_spec = ''
if save_fig:
    file_name_fig = f"{dir_save}timeseries_and_{str_seas}{str_freq}_fit{str_trend}_{len(compare_periods)}periods{str_spec}.{fig_format}"
    ## correct figure size when saving
    save_fig_extact_size(
        fig,
        file_name_fig,
        figsize_cm = (figW_temp, figH_temp),  # Width & height in cm
        margins_cm=(1.5, 1.3, 0.4, 1.3),  # (left, bottom, right, top) in cm
        fig_dpi=fig_dpi,
        fig_format=fig_format)

### Time series figure after 2020 but with CAMS data in addition <a class="anchor" id="fig_ts"></a>
Note that our observatoins are 1h data, CAMS is 3h (or 6h for CH4)

In [ ]:
# Select data to plot

# species to plot
species_aer = 'black_carbon'
my_species = [ 'CO2', 'CH4', 'O3', 'CO', species_aer]

# GHG data (with fire events/outliers included or not)
ds_data = ds_all.where(ds_all.dataset.isin(my_species), drop=True).sel(time=slice(time1, time2))
ds_data_rem_out = ds_all_rem_out.where(ds_all_rem_out.dataset.isin(my_species), drop=True).sel(time=slice(time1, time2))

# aerosol data (with fire events/outliers included or not)
# when using full aerosol data: 
# Define which wavelenght to use for black carbon data
# lambda_bc = 880 # wavelength for black carbon
# aer_sel = ds_aerosols.sel(lambda_abs=lambda_bc,time=slice(time1,time2))[species_aer] /1000 #convert to µg/m³
# aer_sel_rem_out = ds_aerosols_rem_out.sel(lambda_abs=lambda_bc,time=slice(time1,time2))[species_aer] /1000 #convert to µg/m³

# when using prepared black carbon data:
aer_sel = ds_aerosols
aer_sel_rem_out = ds_aerosols_rem_out


In [ ]:
# Define again fit paramaters for the curve fitting, for all species (including black carbon)
fit_properties = {}
for dataset in my_species:
    #dataset_name = dataset.values.item()  # Convert numpy array to a hashable type
    fit_properties[dataset] = fit_params_defaults.copy()

In [ ]:
## Load the selected cams data at Mt. Kenya
cams_best_grid = xr.open_dataset(f"{data_path}/level3/cams/cams_best_grid_merged_MKN.nc") #selection where 2 invgg periods (different resolution) were merged

## Remove cams products that I don't want to plot
cams_best_grid = cams_best_grid.drop_sel(dataset=["co2_egg4","ch4_egg4"])
cams_best_grid = cams_best_grid.sel(time=slice(time1, time2))


In [ ]:
# Read in CAMS Black carbon data (from EAC4)
cams_eac4_aer = xr.open_dataset(f"{external_data_path}/CAMS/cams_final_data/cams_eac4_aerosols_2020_2024_MKN.nc")
## Select the same grid as for CO -EAC4 data
eac_lat, eac_lon, eac_lev = [
    cams_best_grid.sel(dataset="co_eac4")[var].values # use the same pressure level as for CO
    for var in ["latitude", "longitude", "level"]
]

cams_eac4_aer_sel = cams_eac4_aer.sel(latitude=eac_lat,longitude=eac_lon, level=eac_lev,drop=True)

In [ ]:
# CAMS black carbon data (sum of hydrophilic and hydrophobic)
# Note: CAMS data is in kg/kg, so I need to convert it to µg/m³
cams_bc = cams_eac4_aer_sel['aermr09']  # Hydrophilic Black Carbon Aerosol Mixing Ratio (in kg/kg)
cams_bc += cams_eac4_aer_sel['aermr10'] # Hydrophobic Black Carbon Aerosol Mixing Ratio (in kg/kg)

In [ ]:
# get EAC4 data for temperature
cams_eac4 = xr.open_dataset(f"{external_data_path}/CAMS/cams_final_data/cams_eac4_2003_2024_MKN.nc")
cams_eac4_sel = cams_eac4.sel(latitude=eac_lat,longitude=eac_lon, level=eac_lev,drop=True)

In [ ]:
## Convert Black carbon MMR to VMR
# determine time series of air density
# rho_d = p/(T*R) # in kg/m³
R_dry_air = 287.052874 # J/kg/K
rho_d = eac_lev * 100 / (cams_eac4_sel['t'] * R_dry_air)# in kg/m³

cams_bc_conc = cams_bc * rho_d * 1e9 # in kg/m³ - >convert to µg/m³ with *1e9

In [ ]:
# --- helper: get start and end DOY for a month ---
import calendar
def month_to_doy(month, leap=False):
    """Return (start_doy, end_doy) for a given month number (1-12)."""
    days_in_month = [calendar.monthrange(2000 if leap else 2001, m)[1] for m in range(1, 13)]
    cumdays = np.cumsum([0] + days_in_month)  # cumulative sum, length 13
    start = cumdays[month - 1] + 1
    end = cumdays[month]
    return start, end

In [ ]:
ds_data_rem_out.sel(dataset='CO2')["value"].resample(time='1h').mean()

In [ ]:
### Figure to compare cams with newest measurement data, including line fits
# it only works with the newest data from 2020 and not with flask data!
# The reason is because we are looping through the species, so the dataset and sepcies name has to be the same (e.g. 'CO2', the dataset 'CO2_flask' would not work)


##---- Plotting definitions ----##
figW_temp = figW3
figH_temp = figH2*2 #figH3
# plot seasonal or daily cycles
plot_seas = True  # if false, plot daily cycle instead of seasonal
seas_freq = "dayofyear"  # 'month' or 'dayofyear'
with_trend = False  # if true, plot detrended data, otherwise detrend seasonal cycle but dont plot it as extra line in the time series plot
with_outliers = True  # use data where outliers are removed or not
with_cams_points = True
with_ae_instruments = True # plot the 2 Aerosol instruments (AE31 and AE33) separately or not

temporal_resolution_measurements = '3h' # default: '1h'

default_colors = ["C0", "C1", "C2", "C3"]
color_iterator = itertools.cycle(default_colors)

# for with_outliers in [False, True]: # if outliers should be once plotted and once not: use the following lines and indent all the rest
# dataset to use:
# if with_outliers:
#     ds_data = ds_all #normal measurement data (outliers not removed)
#     ds_aer = ds_aerosols #normal aerosol data (outliers not removed)
# else:
#     ds_data = ds_all_rem_out  # use data with removed outliers!!
#     ds_aer = ds_aerosols_rem_out # use aerosol data with removed outliers
# # only select data where the species is in my_species:
# ds_data = ds_data.where(ds_data.dataset.isin(my_species), drop=True).sel(time=slice(time1, time2))
# aer_sel = ds_aer.sel(lambda_abs=880,time=slice(time1,time2))['black_carbon'] # select black carbon data at 880nm


##--- Start figure  ---##
# Make a mosaic grid, where each subplot is called either the species (e.g. CO2) or the species seasonal cycle (ew.g. CO2_seas)

mosaic_grids = [[m, m + "_seas"] for m in my_species]  # put [] around to start new row
gs_kw = dict(width_ratios=[3, 1])  # column and width ratios
subfig_height = (
    2.5 if len(my_species) == 1 else 2
)  # height of each subplot, larger when we would plot only 1 species
fig, axd = plt.subplot_mosaic(
    mosaic_grids,
    gridspec_kw=gs_kw,
    figsize=(cm2inch((figW_temp, figH_temp))),
    layout="constrained",
)

# Filter the data

# Define figure properties
fig_properties = {
    "color": "C0",  # next(color_iterator),
    "marker": ".",
    "ls": "-",
    "zorder": 2,
}

## Loop through all datasets and plot the data
for i, dataset in enumerate(my_species):
    # Get the data for the current dataset
    if dataset == "black_carbon":
        data = aer_sel_rem_out["black_carbon_mean"].resample(time=temporal_resolution_measurements).mean()
        species = "black_carbon"
        species_str = "Black carbon"
    else:
        data = ds_data_rem_out.sel(dataset=dataset)["value"].resample(time=temporal_resolution_measurements).mean()
        species = str(ds_data_rem_out.sel(dataset=dataset).species.values)
        species_str = re.sub(r"(\d+)", r"$_\1$", species)
    unit = (
        "$\mu g/$m$^3$"
        if species == "black_carbon"
        else f"{ds_data.sel(dataset=dataset).unit.values}"
    )

    if data.size > 0 and np.all(np.isnan(data)) == False:  # if we have data
        # ===================================
        # Define fit properties for this dataset
        # Perform the curve fitting and obtain the interpolated time axis
        filt, df_interp, ds_interp = run_curve_fit.run_ccgfilter(
            ds=data, dataset_str=dataset, t1=time1, t2=time2, **fit_properties[dataset]
        )

        # ===================================
        # Plot the data and the fit
        # normal properties
        legend_entry = "Measurements"
        fig_properties["marker"] = "."
        fig_properties["color"] = "C0"
        fig_properties["zorder"] = 5

        # ===================================
        ##--- Time series figure
        ax_ts = axd[species]
        ## Seasonal cycle figure
        ax_seas = axd[species + "_seas"]

        # ===================================
        ## Plot the fire events in lighter blue
        # manually set the ylims for CO and BC, even though that cuts off the fire events
        if species in ["CO", "black_carbon"]:
            if species == "CO":
                data_with_events = ds_data.sel(dataset=dataset)["value"]
                ymax = 350

            if species == "black_carbon":
                data_with_events = aer_sel[
                    "black_carbon_ae31"
                ]  # we only had AE1 data at the fire events in 2021/2022
                ymax = 2.3  # µg/m³

            # cutoff the fire event data and indicate it with an arrow
            fire_event_color = getattr(col_contr_medium, "light_blue")
            # event 2022
            data_with_events.sel(time=fire_dates_Mar2022).plot(
                ax=ax_ts,
                color=fire_event_color,
                marker=".",
                ls="",
                label="Fire events",
                zorder=5,
            )

            # # event 2024
            # data_with_events.sel(time=fire_dates_JA2024).plot(
            #     ax=ax_ts,
            #     color='red',
            #     marker="d",
            #     ls="",
            #     label="Fire events",
            #     zorder=10,
            # )

            # event 2021 (cut off)
            data_with_events.sel(time=fire_dates_Mar2021).plot(
                ax=ax_ts, color=fire_event_color, marker=".", ls="", label="", zorder=5
            )
            fire_outliers = data_with_events.where(data_with_events > ymax, drop=True)
            y_highest = fire_outliers.max().values
            outlier_times = fire_outliers.where(fire_outliers, drop=True).time
            fire_outliers.loc[dict(time=outlier_times)] = (
                ymax  # set outliers to max. allowed value
            )
            ax_ts.set_ylim(top=ymax)
            # fire_outliers.plot(ax=ax_ts, color='red', marker='^', ls='', label='Extreme event (cut-off)',zorder=5) # plot all outlier values set to ymax
            # only plot one single triangle at the time of outlier event at ymax
            central_event_time = pd.Timestamp(
                fire_outliers.time[int(len(fire_outliers.time) / 2)].values
            )
            shifted_time = central_event_time + pd.DateOffset(
                months=1
            )  # shift the triangle and label by 1 month
            ax_ts.plot(
                shifted_time,
                ymax - ymax * 0.08,
                marker="^",
                color=fire_event_color,
                ls="",
                zorder=5,
                #markersize=4,
            )

            # Add annotation for the outlier
            ax_ts.annotate(
                f"  up to {y_highest:.0f}{unit} (cut off event)",
                xy=(shifted_time, ymax),
                xytext=(shifted_time, ymax - ymax * 0.1),
                fontsize="x-small",
                color=fire_event_color,
            )

            ax_ts.legend(loc="upper right")

        # ===================================
        ## Plot ozonesonde data in addition
        if species == "O3":
            o3sonde_daily.plot(
                ax=ax_ts,
                label=f"ozonesonde ({plev}hPa)",
                marker="X",
                ls="",
                color="k",
                zorder=10,
            )
            #ax_ts.legend(loc="upper center", ncols=2)
            ax_ts.legend(loc=(0.6,0.85), ncols=2)

        # ===================================
        ## Plot the measurement data
        # legend_entry = f"curve fit ({fit_properties[dataset]['shortterm']} days)"
        legend_entry = "Measurements"

        # For black carbon: plot the single AE31 and AE33 separately. The fit line can be the mean value
        if species == "black_carbon":
            # plot mean fit, but without data points:
            plot_data(
                data,
                ds_interp,
                label_meas="",
                label_fit="",
                ax=ax_ts,
                detrend=with_trend,
                plot_measurements=False,
                **fig_properties,
            )

            # plot the black carbon, if desired plot the AE31 and AE33 instruments separately
            if with_ae_instruments:
                # AE31 data
                data_ae = aer_sel_rem_out["black_carbon_ae31"]
            else:
                data_ae = data

            plot_data(
                data_ae,
                ds_interp,
                label_meas="AE31",
                label_fit="",
                ax=ax_ts,
                detrend=with_trend,
                plot_fit=False,
                **fig_properties,
            )
            
            if with_ae_instruments:
                # AE33 data 
                fig_properties["marker"] = "x"
                fig_properties["color"] = "k"
                # fig_properties['zorder'] = 4
                data_ae33 = aer_sel_rem_out["black_carbon_ae33"]
                plot_data(
                    data_ae33,
                    ds_interp,
                    label_meas="AE33",
                    label_fit="",
                    ax=ax_ts,
                    detrend=with_trend,
                    plot_fit=False,
                    **fig_properties,
                )
                # black carbon legend
                #ax_ts.legend(loc="upper right", ncols=3)
                ax_ts.legend(loc=(0.74,0.6), ncols=1)

                # reset fig properties to default
                fig_properties["color"] = "C4"
                fig_properties["marker"] = "."
        else:
            plot_data(
                data,
                ds_interp,
                label_meas=legend_entry,
                label_fit="",
                ax=ax_ts,
                detrend=with_trend,
                **fig_properties,
            )
        # ===================================
        ##--- Seasonal cycle
        freq = (
            seas_freq if plot_seas else "hour"
        )  # for seasonal cycle, use 'dayofyear' (smoother line) or 'month', for daily cycle (if plot_seas=False) use 'hour'
        # no markers for seasonal cycle
        fig_properties["marker"] = ""  # no marker for seasonal cycle
        pl_seas = plot_cycle(
            data,
            ds_interp,
            species,
            freq=freq,
            ax=ax_seas,
            with_trend=with_trend,
            **fig_properties,
        )

        # Add shading for different seasons
        xvals = ds_interp["seasonal_detrend"].groupby(f"time.{freq}").mean()[freq]
        for season, seas_m in my_seasons.items():
            doy1, _ = month_to_doy(seas_m[0])
            _, doy2 = month_to_doy(seas_m[-1])
            ax_seas.axvspan(doy1-1, doy2, color=seas_cols[season], alpha=0.2, label=season, lw=0, antialiased=False )
            
            # For season labels
            
            if species == my_species[0]: # first axes
                # Add text label centered horizontally in the span
                x_center = (doy1 + doy2) / 2
                # y_pos = -ax_seas.get_ylim()[1] * 0.9   # -90% of the y-axis height (below zero)
                # ax_seas.text(
                #     x_center, y_pos, season,
                #     ha="center", va="center", color=seas_cols[season],alpha=.9
                # )
                ax_seas.text(
                    x_center, 0.02,  # y=0 in axis coordinates
                    season,
                    ha="center",
                    va="bottom",  # align text above this y-position
                    color=seas_cols[season],
                    alpha=0.7,
                    transform=ax_seas.get_xaxis_transform()  # uses axis coords for y
                )

        # ===================================
        # ---- plot the CAMS data ----#

        # Select the corresponding CAMS data (no data for Black carbon)
        if dataset == "black_carbon":
            data_cams = cams_bc_conc
            species = "black_carbon"
        else:
            cams_datasets_sel = [
                cams_sel
                for cams_sel in cams_best_grid.dataset.values
                if f"{species.lower()}_" in cams_sel
            ]
            data_cams = cams_best_grid.sel(dataset=cams_datasets_sel)[
                "value"
            ].squeeze()  # select the species from cams data

        # Perform the curve fitting for CAMS data and obtain the interpolated time axis
        filt, df_interp, ds_interp = run_curve_fit.run_ccgfilter(
            ds=data_cams,
            dataset_str=species,
            t1=time1,
            t2=time2,
            **fit_properties[species],
        )

        fig_properties["color"] = "C2"  # CAMS color
        # legend_entry = f"CAMS fit ({fit_properties[species]['shortterm']} days)"
        legend_entry = "CAMS"
        if with_cams_points:
            fig_properties["marker"] = "."
        else:
            fig_properties["marker"] = (
                ""  # dont show the CAMS markers, show only the fit
            )

        # ===================================
        plot_data(
            data_cams,
            ds_interp,
            label_meas=legend_entry,
            label_fit=f"fit ({fit_properties[dataset]['shortterm']} days)",
            ax=ax_ts,
            detrend=with_trend,
            **fig_properties,
        )

        # ===================================
        # plot seasonal cycle
        freq = (
            seas_freq if plot_seas else "hour"
        )  # for seasonal cycle, use 'dayofyear' (smoother line) or 'month', for daily cycle (if plot_seas=False) use 'hour'
        # no markers for seasonal cycle
        fig_properties["marker"] = ""  # no marker for seasonal cycle
        # fig_properties['zorder'] = 4
        pl_seas = plot_cycle(
            data_cams,
            ds_interp,
            species,
            freq=freq,
            ax=ax_seas,
            with_trend=with_trend,
            label="",  #'cams'
            **fig_properties,
        )
        # Cams and measurements legend only in first subplot:
        if i == 0:
            ax_ts.legend(loc="upper center", ncols=3)

        # ===================================
        # ---- figure properties ----#

        # axes properties time series
        label = (
            f"BC ({unit})" if species == "black_carbon" else f"{species_str} ({unit})"
        )
        ax_ts.set_ylabel(label)
        # ax_ts.set_xlim(np.array([t1, t2], dtype="datetime64"))
        ax_ts.spines["top"].set_visible(False)
        ax_ts.spines["right"].set_visible(False)
        ax_ts.set_title("")
        # be sure to have same x-axis for all plots
        ax_ts.set_xlim([pd.to_datetime(time1), pd.to_datetime(time2)])

        # subplot labelling
        #ax_ts.set_title(f"({alphabets[i]}) {species_str}", loc="left")
        #ax_seas.set_title(f"({alphabets[i+len(species)]})", loc="left")
        for ax_sel, igap in zip([ax_ts, ax_seas],[0, len(my_species)]):
            ax_sel.text(
                0.015,
                1,
                (f"({alphabets[i+igap]}) {species_str}"),
                transform=ax_sel.transAxes,
                bbox=dict(fc="white", ec="none", alpha=0),
                zorder=2,
                fontsize="small",
                verticalalignment="top"
            )

        if species != my_species[-1]:  # all except lowest axis
            plt.setp(
                ax_ts.get_xticklabels(), visible=False
            )  # remove xticklabels except for lowest plot
            ax_ts.xaxis.set_major_locator(mdates.YearLocator())
            ax_ts.xaxis.set_minor_locator(
                mdates.MonthLocator(bymonth=[1, 7])
            )  # Jan and July
            ax_ts.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10])) 
            ax_ts.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
            ax_ts.set_xlabel("")
        else:  # lowest axis
            ax_ts.set_xlabel("Time")
            # ax_ts.xaxis.set_major_locator(mdates.YearLocator())
            # ax_ts.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
            # ax_ts.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
            ax_ts.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,7])) 
            ax_ts.xaxis.set_major_formatter(custom_month_tick_formatter)
            ax_ts.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
            # ax_ts.xaxis.set_minor_formatter(mdates.DateFormatter('%b'))
            # rotate the xlabels
            # plt.setp(ax_ts.xaxis.get_majorticklabels(), rotation=45)
            # ax_ts.legend(loc='upper left')

        # axes properties seasonal cycle
        # ax_seas.set_yticklabels('') #this removes ylabels for both axes! Therefore better use:
        # ax_seas.tick_params(labelleft=False) #I need the labels if I use other units for the seasoal cycle!!
        ax_seas.set_title("")
        # ax_seas.set_ylabel(f"{species} deviation ({unit})")
        ax_seas.set_ylabel("")
        ax_seas.spines["top"].set_visible(False)
        ax_seas.spines["right"].set_visible(False)
        if plot_seas:
            if freq == "month":
                months = range(1, 13, 3)
                ax_seas.set_xticks(
                    months,
                    [
                        dt.datetime.strptime(str(month), "%m").strftime("%b")
                        for month in months
                    ],
                )
                # rotate the xlabels
                plt.setp(ax_seas.xaxis.get_majorticklabels(), rotation=45)
            elif freq == "dayofyear":
                ax_seas.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13, 3)))
                ax_seas.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
                ax_seas.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
                # rotate the xlabels
                # plt.setp(ax_seas.xaxis.get_majorticklabels(), rotation=45)

        # ax_seas.grid(color='lightgrey',zorder=0,axis='x')
        ax_seas.set_axisbelow(False)

        # different for lowest plot
        if species != my_species[-1]:  # all except lowest axis
            plt.setp(
                ax_seas.get_xticklabels(), visible=False
            )  # remove xticklabels except for lowest plot
            ax_seas.set_xlabel("")
        elif species == my_species[0]: # first axes
            ax_seas.legend(loc='upper right',ncols=2) # legend for season shading. NOT WORKING?
        else:  # lowest axis
            ax_seas.set_xlabel("Months" if plot_seas else "Hours")

# ===================================
## --- Final figure properties ---##
# handles, labels = ax_ts.get_legend_handles_labels()
# # manually adapt legend:
# labels[0] = "1h data"
ax0 = list(axd.values())[0]  # first subplot (upper left)
# if with_trend:
#     ax0.legend(handles, labels, loc="lower right")

# place title on first axis
ax0.set_title(
    f"Mt. Kenya measurements ({temporal_resolution_measurements}) and CAMS data (3h)",
    loc="left",
)
ax1 = list(axd.values())[1]  # second subplot (upper right)
if plot_seas:
    tit_right = "Seasonal cycle (detrended)"
else:
    tit_right = "Diurnal cycle"
ax1.set_title(tit_right, loc="center")
fig.align_ylabels()

str_seas = "seas" if plot_seas else "daily"
str_trend = "_detrended" if with_trend else ""
str_trend = str_trend + f"{fit_properties[dataset]['shortterm']}d"
str_freq = "_" + seas_freq if plot_seas else ""

# if not all species are plotted: adapt name
if len(my_species) < 4:
    str_spec = "_" + "".join(my_species)
else:
    str_spec = ""
# if with_outliers:
#     str_spec += '_with_outliers'
if save_fig:
    # plt.savefig(
    #     f"{dir_save}timeseries_and_{str_seas}{str_freq}_fit{str_trend}{str_spec}.{fig_format}", dpi=fig_dpi
    # )
    file_name_fig = f"{dir_save}timeseries_and_{str_seas}{str_freq}_fit{str_trend}{str_spec}_{temporal_resolution_measurements}.{fig_format}"
    ## correct figure size when saving
    save_fig_extact_size(
        fig,
        file_name_fig,
        figsize_cm=(figW_temp, figH_temp),  # Width & height in cm
        margins_cm=(
            1.5,
            1.3,
            0.4,
            1.3,
        ),  # (left, bottom, right, top) in figure fraction
        fig_dpi=fig_dpi,
        fig_format=fig_format,
    )

In [ ]:
# # check BC Data
# plt.figure()
# aer_sel["black_carbon_mean"].plot()
# aer_sel["black_carbon_mean"].sel(time=fire_dates_JA2024).plot(
#                 color='red',
#                 marker="d",
#             )
# aer_sel["black_carbon_mean"].sel(time='2024-07-15').plot(
#                 color='red',
#                 marker="d",
#             )
# aer_sel["black_carbon_mean"].sel(time='2024-07-27').plot(
#                 color='red',
#                 marker="d",
#             )
# plt.show()

### Diurnal cycles  <a class="anchor" id="fig_diurnal"></a>

In [ ]:
# define which temperature data to plot: 
# use only data starting in 2022 (less data available before)
temperature = ds_meteo['temperature'].sel(time=slice('2022-01-01',time2))

In [ ]:
## Determine average sunrise and sunset times for Mt. Kenya:

# check day vs night data

mknlat, mknlon, mknalt = get_station_coords("MKN")  # Mt. Kenya station coordinates
sun_mkn = Sun(mknlat, mknlon)
# today_sr = sun.get_sunrise_time()
# today_ss = sun.get_sunset_time()
# print('Today at MKN the sun raised at {} and get down at {} UTC'.
#       format(today_sr.strftime('%H:%M'), today_ss.strftime('%H:%M')))
sunrise_times = [sun_mkn.get_sunrise_time(to_datetime(t)) for t in ds_data.time.values]
# transform times to decimal times
sunrise_times_decimal = [t.hour + t.minute/60 for t in sunrise_times]

sunset_times = [sun_mkn.get_sunset_time(to_datetime(t)) for t in ds_data.time.values]
# transform times to decimal times
sunset_times_decimal = [t.hour + t.minute/60 for t in sunset_times]

# make a pandas dataframe with sunrise and sunset times for each timestep
# set time as index
df_suntime = pd.DataFrame(index=pd.DatetimeIndex(pd.to_datetime(ds_data.time.values)))
df_suntime['time'] = [t.time() for t in df_suntime.index]
df_suntime['time_sunrise'] = [t.time() for t in sunrise_times]
df_suntime['time_sunset'] = [t.time() for t in sunset_times]
df_suntime['is_day'] = (df_suntime['time'] > df_suntime['time_sunrise']) & (df_suntime['time'] < df_suntime['time_sunset'])
df_suntime

## Mean sunrise and sunset times: 

mean_sunrise = (np.mean(sunrise_times_decimal))
mean_sunset = (np.mean(sunset_times_decimal))

In [ ]:
## Group diurnal cycles by season

from labellines import labelLine, labelLines
import matplotlib.patheffects as pe

# species to plot
species_aer = 'black_carbon'
my_species = ['CO2', 'CH4', 'O3', 'CO', species_aer, 'T']



##---- Plotting definitions ----##
figW_temp = figW2 #figW3
figH_temp = figH2 #*1.5

freq = "hour"

# dont use all cams-products
cams_mask = ~np.isin(cams_best_grid["dataset"], "co2_egg4", "ch4_egg4")
cams_use = cams_best_grid.sel(dataset=cams_mask)
# Or: use all:
# cams_use = cams_best_grid


# function to plot each subplot
def plot_data_diurnal(ds, ax, label, col, plot_std=False):
    # other option: use moving-averaged mean:
    #mw_temp = 30
    # ds = ds.rolling(time=mw_temp, center=True, min_periods=mw_temp / 2).mean()
    ds_grouped = ds.groupby(f"time.{freq}").mean()
    ds_grouped.plot(
        ax=ax,
        ls="-",
        marker="",
        color=col,
        markeredgewidth=0,
        # alpha=alpha,
        # markersize=ms,
        # label=f"{label} {str(ds.dataset.values)}",
        label=f"{label}",
    )
    delta_diurnal = (ds_grouped.max()-ds_grouped.min())
    print(f"Mean ampltitude: {delta_diurnal:.2f}")
    
    # Add standard deviation
    if plot_std:
        std = ds.groupby(f"time.{freq}").std() # standarddeviation of the grouped values of the measurements
        # add stdev
        ax.fill_between(
            ds_grouped[freq],
            ds_grouped,
            ds_grouped + std,
            alpha=0.3,
            label=f"std. dev.",
            color=col,
        )
        ax.fill_between(
            ds_grouped[freq],
            ds_grouped,
            ds_grouped - std,
            alpha=0.3,
            label=f"std. dev.",
            color=col,
        )


fig, axs = plt.subplots(
    2, 3, sharex=True, figsize=(figW_temp, figH_temp), layout="constrained"
)
axs = axs.flatten()
for i, (s, ax) in enumerate(zip(my_species, axs)):
    print(s)
    if s == 'T':
        #plot temperature
        ds_sel = temperature # temperature at 2m
        species= 'T'
        species_str = "Temperature"
        unit = "°C"
    elif s == 'black_carbon':
        ds_sel = aer_sel_rem_out['black_carbon_mean']
        species = 'black_carbon'
        species_str = "Black carbon"
        unit = '$\mu g/$m$^3$'
    else:
        ds_sel = ds_data_rem_out.sel(dataset=s)["value"]
        species = str(ds_data_rem_out.sel(dataset=s).species.values)
        # put $_$ in front of numbers:
        species_str = re.sub(r"(\d+)", r"$_\1$", species)
        unit = f"{ds_data.sel(dataset=s).unit.values}"
    

    #ds_sel = data.sel(time=slice(time1, time2)).where(data.species == s, drop=True)
    # select cams datasets that exist for this species
    cams_datasets_sel = [
        cams_sel for cams_sel in cams_use.dataset.values if f"{s.lower()}_" in cams_sel
    ]
    cams = cams_use.sel(dataset=cams_datasets_sel, time=slice(time1, time2))

    # plot obs
    # Loop through seasons
    for season, season_months in my_seasons.items():
        print(season)
        # ds_sel_season = ds_sel.groupby("time.season")[season]
        ds_sel_season = ds_sel.where(
            ds_sel.time.dt.month.isin(season_months), drop=True
        )
        plot_data_diurnal(
            ds_sel_season,
            ax,
            f"{season}",
            col=seas_cols[season],
            plot_std=False,  # plot standard deviation
        )


    ##-- Highlight sunrise and sunset times
    # sunrise and sunset times
    ax.axvspan(mean_sunrise, mean_sunset, color="gold", alpha=0.1)


    ##-- axes properties
    # set minor xticks every 3 hours and major ticks every 6 hours
    ax.set_xlim(0, 26.999)  # set xlims a bit larger to have space for the labels
    ax.xaxis.set_major_locator(ticker.MultipleLocator(6))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(3))

    ax.set_xlabel("")
    ax.set_xlabel("Hour of day (UTC)")
    ax.set_title("")
    #ax.set_title(f"({alphabets[i]}) {ds_sel.species[0].values}", loc="left")
    ax.set_title(f"({alphabets[i]}) {species_str}", loc="left")

    ax.set_ylabel(f"{species_str} ({unit})")


    # Labels/legend

    # labels on each line:
    # ax.legend()
    xval_label = 22.8 # x-value for the label, right border of plot
    label_pos = 'center'
    # Manually adapt label positions for cases with overlap
    if s == 'CO2':
        yoffsets = [-0.3, 0.1, 0.05, -0.2] # JF, MAM, JJAS, OND slightly lower or higher
    elif s == 'O3':
        yoffsets = [-0.1, 0, 0, 0] # JF, MAM, JJAS, OND slightly lower or higher     
    elif s == 'CH4':
        yoffsets = [0, 0.2, 0, 0] # JF, MAM, JJAS, OND slightly lower or higher     
    elif s == 'CO':
        yoffsets = [-1.5, 0, 2.5, 0] # JF, MAM, JJAS, OND slightly lower or higher    
    elif s == 'black_carbon':
        yoffsets = [0, 0, 0, 0.02] # JF, MAM, JJAS, OND slightly lower or higher    
    elif s == 'T':
        #yoffsets = [-0.3, 0.6, 0.27, 0.8] # with 2023-2024 data only
        yoffsets = [-0.3, 0, 0, 0.5] # JF, MAM, JJAS, OND slightly lower or higher
    else:
        yoffsets = [0, 0, 0, 0]
    labels = labelLines(
        ax.get_lines(),
        align=False,
        xvals=[xval_label, xval_label, xval_label, xval_label], # adapt label position
        fontsize='xx-small',
        va = label_pos,
        ha = 'left',
        #backgroundcolor="none",
        yoffsets=yoffsets,
        #shrink_factor=0.6,
    )  # legend on lines, ,xvals=[24.5,24.5,24.5,24.5]
    
    # explicitly set transparent bbox
    for lbl in labels:
        #lbl.set_bbox(dict(facecolor="none", edgecolor="none"))
        lbl.set_path_effects([pe.Normal()]) #remove the default bounding box around the text
        # slightly shift all labels in x direction
        lbl.set_x(lbl.get_position()[0] + .4)
    if s == 'T': # not enough space for labels
        # shift labels manually in x direction
        xshifts = [2.5, 0, -0.1, 0] # JF, MAM, JJAS, OND
        for lbl, dx in zip(labels, xshifts):
            lbl.set_x(lbl.get_position()[0] + dx)

# align ylabels
fig.align_ylabels()
plt.suptitle(f"Diurnal cycles for different species and seasons", y=1.03, x=0.09, ha='left')

if save_fig:
    file_name_fig = f"{dir_save}diurnal_cycles.{fig_format}"
    ## correct figure size when saving
    save_fig_extact_size(
        fig,
        file_name_fig,
        figsize_cm=(figW_temp, figH_temp),  # Width & height in cm
        margins_cm=(0.5, 0.5, 0.5, 0.5),  # (left, bottom, right, top) in cm
        fig_dpi=fig_dpi,
        fig_format=fig_format,
    )

In [ ]:
    # sunrise and sunset times
    ax.axvspan(mean_sunrise, mean_sunset, color="gold", alpha=0.1)


    ##-- axes properties
    # set minor xticks every 3 hours and major ticks every 6 hours
    ax.set_xlim(0, 26.9)  # set xlims a bit larger to have space for the labels
    ax.xaxis.set_major_locator(ticker.MultipleLocator(6))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(3))

    ax.set_xlabel("")
    ax.set_xlabel("Hour of day (UTC)")
    ax.set_title("")
    #ax.set_title(f"({alphabets[i]}) {ds_sel.species[0].values}", loc="left")
    ax.set_title(f"({alphabets[i]}) {species_str}", loc="left")

    ax.set_ylabel(f"{species_str} ({unit})")


    # Labels/legend

    # labels on each line:
    # ax.legend()
    xval_label = 22.8 # x-value for the label, right border of plot
    label_pos = 'center'
    # Manually adapt label positions for cases with overlap
    if s == 'CO2':
        yoffsets = [-0.6, 0.1, 0.05, -0.4] # JF, MAM, JJAS, OND slightly lower or higher
    elif s == 'O3':
        yoffsets = [-0.1, 0, 0, 0] # JF, MAM, JJAS, OND slightly lower or higher     
    elif s == 'CO':
        yoffsets = [-2, 0, 2.5, 0] # JF, MAM, JJAS, OND slightly lower or higher    
    elif s == 'black_carbon':
        yoffsets = [0, -0.02, 0, 0.02] # JF, MAM, JJAS, OND slightly lower or higher    
    elif s == 'T':
        yoffsets = [-0.3, 0.6, 0.27, 0.8] # JF, MAM, JJAS, OND slightly lower or higher
    else:
        yoffsets = [0, 0, 0, 0]
    labelLines(
        ax.get_lines(),
        align=False,
        xvals=[xval_label, xval_label, xval_label, xval_label], # adapt label position
        fontsize='xx-small',
        va = label_pos,
        ha = 'left',
        backgroundcolor="none",
        yoffsets=yoffsets,
        #shrink_factor=0.6,
    )  # legend on lines, ,xvals=[24.5,24.5,24.5,24.5]


# align ylabels
fig.align_ylabels()
plt.suptitle(f"Diurnal cycles for different species and seasons")

#if save_fig:
file_name_fig = f"{dir_save}diurnal_cycles.{fig_format}"
## correct figure size when saving
save_fig_extact_size(fig,
    file_name_fig,
    figsize_cm=(figW_temp, figH_temp),  # Width & height in cm
    margins_cm=(0.5, 0.5, 0.5, 0.5),  # (left, bottom, right, top) in cm
    fig_dpi=fig_dpi,
    fig_format=fig_format,
)

## Inventory plots

### Read in inventory data
1) fire data
2) EDGAR CO data
3) EDGAR CH4 data
4) WetCHARTs CH4 data

Only select 1 specific year and plot the averaged emissions (otherwise it gets too crowded)

In [ ]:
inventory_year = 2022

#### 1. Fire data

In [ ]:

import analyses.input.read_cams as read_cams

## Read in CAMS GFAS fire data
dir_data_cams = "/input/ECMWF/CAMS/GFAS_africa/" # path to monthly fire data (here on ddm)
#cams_gfas = read_cams.read_cams_gfas(dir_data_cams).sel(time=slice(time1, time2))
#only select desired year:
cams_gfas = read_cams.read_cams_gfas(dir_data_cams)
cams_gfas = cams_gfas.where(cams_gfas.time.dt.year == inventory_year,drop=True)
cams_gfas_temporal_mean = cams_gfas.mean(dim='time')


#### 2.+3. EDGAR data

In [ ]:
file_path = Path(external_data_path) / f"EDGAR/EDGAR_africa_grouped/"

# inv_edgar_co = xr.open_dataset(file_path/f"CO_{inventory_year}.nc")
# inv_edgar_bc = xr.open_dataset(file_path/f"BC_{inventory_year}.nc")
# inv_edgar_ch4 = xr.open_dataset(file_path/f"CH4_{inventory_year}.nc")
    
edgar_datasets = {
    "CO": xr.open_dataset(file_path / f"CO_{inventory_year}.nc"),
    "BC": xr.open_dataset(file_path / f"BC_{inventory_year}.nc"),
    "CH4": xr.open_dataset(file_path / f"CH4_{inventory_year}.nc"),
}

for key in edgar_datasets:
    edgar_datasets[key] = edgar_datasets[key] / (3600 * 24 * DAY_PER_YR) / edgar_datasets[key]['cell_area']  # kg/yr/cell (emission units emiproc) to flux in kg/m2/s

# final inventory data, remove agriculture emissions from CO and BC
inv_edgar_total_co = edgar_datasets["CO"]["emi_CO_all_sectors"] - edgar_datasets["CO"]["CO_agriculture"] #remove agriculture from CO and BC
inv_edgar_total_bc = edgar_datasets["BC"]["emi_BC_all_sectors"] - edgar_datasets["BC"]["BC_agriculture"] #remove agriculture from CO and BC
inv_edgar_total_ch4 = edgar_datasets["CH4"]["emi_CH4_all_sectors"]


# # Wetcharts: 
# wetchart_path = Path(external_data_path + "/WetCHARTs/")
# file_path =  wetchart_path / f"WetCHARTs_v1_3_3_{inventory_year}.nc"
# inv_wetcharts_ch4 = WetCHARTs(
#     file_path,
#     model=None, #use mean of all models
#     category="wetland_emissions",
#)

#### 4. Wetchart data

In [ ]:
# WETCHARTS data
file_path = Path(external_data_path) / f"WetCHARTs/monthly/"
pattern = file_path / f"{inventory_year}*.nc"
matching_files = list(file_path.glob(f"{inventory_year}*.nc"))

if matching_files:
    print(f"Use inventory of {inventory_year}: {file_path}")
    inventory_full = xr.open_mfdataset(f"{str(file_path)}/{inventory_year}*.nc")
else:
    print(f"Inventory for {inventory_year} not found. Trying previous year...")

# convert kg/h/cell to kg/m2/s
inv_wetcharts_ch4 = (inventory_full["CH4_wetland_emissions"] / (3600 ) / inventory_full["cell_area"]).compute()  # kg/h/cell to kg/m2/s
inv_wetcharts_ch4_mean = inv_wetcharts_ch4.mean(dim='time')  # mean over time (here 1 year)



In [ ]:
# ## Reading in the downloaded files as inventories
# # Wetcharts has to be downloaded separately

# #for yr in years:
# inventory_dir = Path(external_data_path) / f"EDGAR/{inventory_year}"

# inv_edgar_ch4 = EDGARv8(inventory_dir / f"EDGAR_2024_GHG_CH4_{inventory_year}_*.nc")

# inv_edgar_co = EDGARv8(inventory_dir / f"v8.1_FT2022_AP_CO_{inventory_year}*.nc")

# inv_edgar_bc = EDGARv8(inventory_dir / f"v8.1_FT2022_AP_BC_{inventory_year}*.nc")



In [ ]:
## previously, I used the following function to read in the inventories, but now I do it with emiproc
# def read_edgar_files(dir_data, file_name, emission_source, yr1, yr2, return_mean=False):

#     ## Read all yearly inventories and save as a dict
#     print(f"for year {yr1} to {yr2}")
#     file_pattern = f"{dir_data}/{file_name}*_{emission_source}_flx.nc"
#     print("Files:", file_pattern)

#     # Get a sorted list of filenames
#     file_list = sorted(glob.glob(file_pattern))

#     # Open datasets separately, assigning a new time coordinate
#     years = range(yr1,yr2+1)
#     inventory_years = {}
#     for file, year in zip(file_list, years):
#         ds = xr.open_dataset(file)  # Open dataset
#         ds = ds.expand_dims({'time': [pd.Timestamp(f"{year}-01-01")]})  # Add time dimension
#         #datasets.append(ds)
#         inventory_years[year] = ds
#     if not inventory_years:
#         print("No inventory files found!")
#     else:
#         return inventory_years if return_mean == False else xr.concat(list(inventory_years.values()), dim='time').mean(dim='time')


# ## Read in EDGAR CO data
# dir_data_edgar_co = "/input/EDGAR/v8.1/CO" # path to monthly fire data (here on ddm)
# edgar_file_name = "v8.1_FT2022_AP_CO"

# edgar_co_total = read_edgar_files(dir_data_edgar_co,edgar_file_name, "TOTALS", inventory_year, inventory_year, return_mean=True)
# edgar_co_agri = read_edgar_files(dir_data_edgar_co,edgar_file_name, "AWB", inventory_year, inventory_year, return_mean=True) # AWB = Agricultural waste burning
# edgar_co_total_no_awb = edgar_co_total - edgar_co_agri

# ## Read in EDGAR CH4 data
# dir_data_edgar_ch4 = "/project/leob/GAW/Kenya/data/EDGAR/CH4_TOTALS_flx" # path to monthly fire data (here on ddm)
# edgar_file_name = "EDGAR_2024_GHG_CH4"

# # edgar_ch4_total = read_edgar_files(dir_data_edgar_ch4,edgar_file_name, "TOTALS", inventory_year, inventory_year, return_mean=True)

# # Wetcharts: 
# wetchart_path = Path(external_data_path + "/WetCHARTs/")
# file_path =  wetchart_path / f"WetCHARTs_v1_3_3_{inventory_year}.nc"
# inv_wetcharts_ch4 = WetCHARTs(
#     file_path,
#     model=None, #use mean of all models
#     category="wetland_emissions",
# )



In [ ]:
# ## Read in WetCHARTs CH4 data
# wetcharts_ch4_years = xr.open_dataset("/project/leob/GAW/Kenya/data/WetCHARTs/WetCHARTs_v1_3_3_2020_2022_model_mean.nc") # model mean 
# #only select desired year:
# wetcharts_ch4 = wetcharts_ch4_years.where(wetcharts_ch4_years.time.dt.year == inventory_year,drop=True)
# wetcharts_ch4 = wetcharts_ch4.mean(dim='time')['wetland_CH4_emissions'] # units: mg/m²/day (daily emisssions of methane from wetlands)
# # convert to kg m-2 s-1
# wetcharts_ch4 = wetcharts_ch4 * 1e-6 / (24*3600) # mg/m²/day to kg/m²/s 

In [ ]:
### Define area of interest (nested domain)

lon1, lon2, lat1, lat2 = [0.05,60.05,20.05, -34.05]

#### Plot inventory maps <a class="anchor" id="fig_inventories"></a>

In [ ]:
## Combined figure with inventory maps
##---- Plotting definitions ----##
figW_temp = figW2
figH_temp = figH2

projection = ccrs.PlateCarree()
# min and max values for colorbars:
opt_co = dict(vmin=1e-21, vmax=1e-10, transform=projection)
opt_ch4 = dict(vmin=1e-16, vmax=1e-8, transform=projection)

# =====
fig, axs = plt.subplots(2, 2, subplot_kw=dict(projection=projection))
axs = axs.flatten()

titles = [
    "CO emissions from fires (GFAS)",  # (GFAS, {time1[:4]} to {time2[:4]})
    "Anthropogenic CO emissions (EDGAR)",  # (EDGAR, 2020 to 2022)
    "CH4 from wetlands (WetCHARTs)",  # (WetCHARTs, 2020 to Aug. 2022)
    "Anthropogenic CH4 emissions (EDGAR)",  # (EDGAR, 2020-2022)
]
for i, (ax, tit) in enumerate(zip(axs, titles)):
    ax.coastlines()
    ax.set_extent([lon1, lon2, lat1, lat2], crs=projection)
    ax.set_title(f"({alphabets[i]}) {tit}", loc="left", fontsize="small")


# Full numbers on the log-colorbar
cbar_formatter = LogFormatter(labelOnlyBase=False)

# ===================================
ax0 = axs[0]
ax1 = axs[1]
ax2 = axs[2]
ax3 = axs[3]

# GFAS CO from fire emissions
map = cams_gfas_temporal_mean["cofire"]
map = map.where(
    map != 0, other=np.nan
)  # only plot non-zero values, set all others to nan
h = map.plot(
    ax=ax0,
    # norm=LogNorm(vmin=opt_co["vmin"], vmax=opt_co["vmax"]),
    norm=LogNorm(),
    # cmap="viridis",
    cmap="cmc.lajolla_r",
    transform=opt_co["transform"],
    add_colorbar=False,
    rasterized=True,
)
cbar = mpu.colorbar(
    h, ax0, pad=0.015, shrink=0.2
)  # , label="CO emissions (kg m-2 s-1)"


# edgar CO
map = inv_edgar_total_co
map = map.where(
    map != 0, other=np.nan
)  # only plot non-zero values, set all others to nan
h = map.plot(
    ax=ax1,
    norm=LogNorm(vmin=opt_co["vmin"], vmax=opt_co["vmax"]),
    # norm=LogNorm(vmin=opt_co["vmin"], vmax=opt_co["vmax"]),
    cmap="cmc.lajolla_r",
    transform=opt_co["transform"],
    add_colorbar=False,
    rasterized=True,
)
cbar = mpu.colorbar(h, ax1, pad=0.04, shrink=0.2, label="CO emissions (kg m-2 s-1)")


# wetcharts
h = inv_wetcharts_ch4_mean.plot(
    ax=ax2,
    norm=LogNorm(vmin=opt_ch4["vmin"], vmax=opt_ch4["vmax"]),
    cmap="cmc.davos_r",
    transform=opt_ch4["transform"],
    add_colorbar=False,
    rasterized=True,
)
cbar = mpu.colorbar(
    h,
    ax2,
    pad=0.015,
    shrink=0.2,
)  # same colorbar as next ch4 figure #,label="CH4 emissions (kg m-2 s-1)"


# edgar ch4
map = inv_edgar_total_ch4
map = map.where(
    map != 0, other=np.nan
)  # only plot non-zero values, set all others to nan
h = map.plot(
    ax=ax3,
    norm=LogNorm(vmin=opt_ch4["vmin"], vmax=opt_ch4["vmax"]),
    cmap="cmc.davos_r",
    transform=opt_ch4["transform"],
    add_colorbar=False,
    rasterized=True,
)

cbar = mpu.colorbar(
    h, ax3, pad=0.04, shrink=0.2, label="CH4 emissions (kg m-2 s-1)"
)  # , format = cbar_formatter

# ===================================

mpu.set_map_layout(
    axs, width=figW_temp
)  # respects subplot_adjust-parameters and the figure width

fig.subplots_adjust(left=0.01, right=0.85, bottom=0.01, top=0.95, hspace=0, wspace=0.35)
if is_poster:    
    fig.subplots_adjust(left=0.01, right=0.85, bottom=0.01, top=0.95, hspace=0, wspace=0.2)
# ===================================

fig.suptitle = f"Emission maps for {inventory_year}"

plt.show()
if save_fig:
    file_name_fig = f"{dir_save}/inventory_maps.{fig_format}"
    plt.savefig(
        file_name_fig, dpi=fig_dpi, format=fig_format
    )  # the mpu.map_layout already adjusted the correct figure size!

#### Plot sectorial contributions of the inventory


In [ ]:
### TO DO (not sure yet...)
# anthropogenic sector distribution for the 3 species
# --- Step 1: Extract summed emissions ---
data_dict = {}
for species in ['CO', 'BC', 'CH4']:
    da = edgar_datasets[species].sum(dim="lat").sum(dim="lon")[f"emi_{species}_total"]
    data_dict[species] = da.values

# Categories (same across species)
categories = da['category'].values

# DataFrame: rows = species, cols = categories
df = pd.DataFrame(data_dict, index=categories).T

# Normalized (sector shares)
df_norm = df.div(df.sum(axis=1), axis=0)

# Absolute totals (for annotations)
totals = df.sum(axis=1)

# --- Step 2: Plot normalized stacked bars ---
ax = df_norm.plot(
    kind="bar",
    stacked=True,
    figsize=(8,5),
    colormap="tab20"
)

# Add totals above bars
for i, total in enumerate(totals):
    ax.text(
        i, 1.02,               # x = bar position, y = just above bar (since bars sum to 1)
        f"{total:.2e}",        # scientific notation; change to :.2f if you prefer
        ha="center", va="bottom", fontsize=10, fontweight="bold"
    )

# Labels & title
ax.set_ylabel("Fraction of total emissions")
ax.set_xlabel("Species")
ax.set_title("Relative Sector Shares (with Total Emissions)")
plt.xticks(rotation=0)
plt.legend(title="Category", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Folded time series

### Read in folded CO data (flexpart, done on cscs)

In [ ]:
# --- GFAS ---#
## Read in GFAS-derived flexpart CO:
co_flexpart_gfas_old = xr.open_dataset(
    data_path + "level3/flexpart/weighted_co_ts_values_2020-2023.nc"
) #initial version, with wrong january values (missing inventory of previous 20 days)
co_flexpart_gfas = xr.open_dataset(
    data_path + "level3/flexpart/weighted_co_GFAS_TOTALS_ts_values_2975m_2020-2023.nc"
) #new update with corrected january values (including inventory of previous 20 days)

# rename time for GFAS
co_flexpart_gfas = co_flexpart_gfas.rename({"release_time": "time"})
# GFAS only on 1 level (2975). For consistency, add the release_height coordinate: 
co_flexpart_gfas = co_flexpart_gfas.expand_dims({'release_height':co_flexpart_gfas_old.coords["release_height"]})

# # compare old with new gfas flexpart data => differences in January
# plt.figure()
# co_flexpart_gfas['co_contribution'].sel(release_height=2975).plot(label='new GFAS flexpart data (with correct jan values)')
# co_flexpart_gfas_old['co_contribution'].sel(release_height=2975).plot(label='old GFAS flexpart data (missing jan values)')
# plt.legend()
# plt.show()

# --- EDGAR ----#
co_flexpart_edgar = xr.open_dataset(
    data_path + "level3/flexpart/weighted_co_EDGAR_ALL_ts_values_2020_2024.nc"
)  # EDGAR totals

# co_flexpart_edgar_awb = xr.open_dataset(
#     data_path + "level3/flexpart/weighted_co_EDGAR_AWB_ts_values_2020_2022.nc"
# )  # EDGAR totals
# # co_flexpart_edgar_no_agri = xr.open_dataset(data_path +'level3/flexpart/weighted_co_EDGAR_total_without_agri_ts_values_2975m_2020-2022.nc') # not sure...

#--- 3h ---#
co_flexpart_edgar_3h = xr.open_dataset(
    data_path + "level3/flexpart/weighted_co_EDGAR_ALL_ts_values_3h_2020_2023.nc"
)  # EDGAR totals, 3-hourly
# process to 3-hourly ts (if not done yet in flexpart folding)
co_flexpart_edgar_3h = process_data.flexpart_to_3hourly(co_flexpart_edgar_3h)
#--- 3h ---#

#--- full domain ---#
# Same but for full domain
co_flexpart_edgar_full_domain = xr.open_dataset(
    data_path + "level3/flexpart/weighted_ts_full_domain/weighted_co_EDGAR_ALL_ts_values_2020_2024.nc"
)
co_flexpart_gfas_full_domain = xr.open_dataset(
     data_path + "level3/flexpart/weighted_ts_full_domain/weighted_co_GFAS_ALL_ts_values_2975m_2020-2023.nc") 
co_flexpart_gfas_full_domain = co_flexpart_gfas_full_domain.rename({"release_time": "time"})
# so far, the gfas full domain was only run for one release level. For consistency, add the release_height coordinate: 
co_flexpart_gfas_full_domain = co_flexpart_gfas_full_domain.expand_dims({'release_height':co_flexpart_gfas.coords["release_height"]})
#--- full domain ---#


In [ ]:
#### Read in the background CO data derived from CAMS
co_bg = xr.open_dataset(data_path + "level3/flexpart/MKN_co_BC_CAMS_EAC4_2020_2023.nc")
co_bg_daily = co_bg["CO_bg"].resample(time="1D").mean()

## Adding natural and Anthropogenic CO from EDGAR
co_all = (
    co_bg_daily
    + co_flexpart_gfas["co_contribution"]
    + co_flexpart_edgar["co_contribution"]
)

# #using full domain:
co_all_full_domain = (
    co_bg_daily
    + co_flexpart_gfas_full_domain["co_contribution"] 
    + co_flexpart_edgar_full_domain["co_contribution"]
)  

# remove agiruclture to avoid double counting:
co_all_removed_agri = (
    co_bg_daily
    + co_flexpart_gfas["co_contribution"]
    + co_flexpart_edgar["co_contribution"]
    - co_flexpart_edgar["CO_agriculture"]
)

co_all_removed_agri_full_domain = (
    co_bg_daily
    + co_flexpart_gfas_full_domain["co_contribution"]
    + co_flexpart_edgar_full_domain["co_contribution"]
    - co_flexpart_edgar_full_domain["CO_agriculture"]
)

### Read CH4

In [ ]:
# --- GFAS ---#
## Read in GFAS-derived flexpart ch4
ch4_flexpart_gfas_old = xr.open_dataset(
    data_path + "level3/flexpart/weighted_ch4_ts_values_GFAS_2020-2023.nc"
) #initial version, with wrong january values (missing inventory of previous 20 days)
ch4_flexpart_gfas = xr.open_dataset(
    data_path + "level3/flexpart/weighted_ch4_GFAS_TOTALS_ts_values_2975m_2020-2023.nc"
) #new update with corrected january values (including inventory of previous 20 days)

# rename time for GFAS
ch4_flexpart_gfas = ch4_flexpart_gfas.rename({"release_time": "time"})
# GFAS only on 1 level (2975). For consistency, add the release_height coordinate: 
ch4_flexpart_gfas = ch4_flexpart_gfas.expand_dims({'release_height':ch4_flexpart_gfas_old.coords["release_height"]})


# --- EDGAR ----#
ch4_flexpart_edgar = xr.open_dataset(
    data_path + "level3/flexpart/weighted_ch4_EDGAR_ALL_ts_values_2020_2024.nc"
)

ch4_flexpart_wetcharts = xr.open_dataset(data_path + "level3/flexpart/weighted_ch4_WetCHARTs_wetlands_ts_values_2020_2024.nc")

# 3hourly
ch4_flexpart_edgar_3h = xr.open_dataset(
    data_path + "level3/flexpart/weighted_ch4_EDGAR_ALL_ts_values_3h_2020_2023.nc"
)  # EDGAR totals, 3-hourly, for testing
ch4_flexpart_wetcharts_3h = xr.open_dataset(
    data_path + "level3/flexpart/weighted_ch4_WetCHARTs_wetlands_ts_values_3h_2020_2023.nc"
)  # EDGAR totals, 3-hourly, for testing


# Same but for full domain
ch4_flexpart_edgar_full_domain = xr.open_dataset(
    data_path + "level3/flexpart/weighted_ts_full_domain/weighted_ch4_EDGAR_ALL_ts_values_2020_2024.nc"
)
ch4_flexpart_wetcharts_full_domain = xr.open_dataset(data_path + "level3/flexpart/weighted_ts_full_domain/weighted_ch4_WetCHARTs_wetlands_ts_values_2020_2024.nc")

ch4_flexpart_gfas_full_domain = xr.open_dataset(
     data_path + "level3/flexpart/weighted_ts_full_domain/weighted_ch4_GFAS_ALL_ts_values_2975m_2020-2023.nc"
) 
ch4_flexpart_gfas_full_domain = ch4_flexpart_gfas_full_domain.rename({"release_time": "time"})
# so far, the gfas full domain was only run for one release level. For consistency, add the release_height coordinate: 
ch4_flexpart_gfas_full_domain = ch4_flexpart_gfas_full_domain.expand_dims({'release_height':ch4_flexpart_gfas.coords["release_height"]})


In [ ]:
#### Read in the background CH4 data derived from CAMS
ch4_bg = xr.open_dataset(
    data_path + "level3/flexpart/MKN_ch4_BC_CAMS_GHGINV_v23r1_2020_2023.nc"
)
ch4_bg_daily = ch4_bg["CH4_bg"].resample(time="1D").mean()

ch4_bg = ch4_bg["CH4_bg"]


# ## Adding all contributions to background
ch4_all = (
    ch4_bg_daily
    + ch4_flexpart_gfas["ch4_contribution"] 
    + ch4_flexpart_edgar["ch4_contribution"]
    + ch4_flexpart_wetcharts["ch4_contribution"]
)  # edgar and wetcharts not yet done for 3678mz

##using full domain:
ch4_all_full_domain = (
    ch4_bg_daily
    + ch4_flexpart_gfas_full_domain["ch4_contribution"] 
    + ch4_flexpart_edgar_full_domain["ch4_contribution"]
    + ch4_flexpart_wetcharts_full_domain["ch4_contribution"]
)  

# ## 3h
ch4_all_3h = (
    ch4_bg
    #+ ch4_flexpart_gfas_3h["ch4_contribution"]  # not run yet!!
    + ch4_flexpart_edgar_3h["ch4_contribution"]
    + ch4_flexpart_wetcharts_3h["ch4_contribution"]
)  # edgar and wetcharts not yet done for 3678m

### Read BC data

In [ ]:
## Read in flexpart bc
#--- GFAS ---#
## Directly convert folded Black carbon from kg/m³ to µg/m³

bc_flexpart_gfas_old = xr.open_dataset(
    data_path + "level3/flexpart/weighted_bc_ts_values_2020-2023.nc"
) * 1e9  # convert from kg/m³ to µg/m³ #initial version, with wrong january values (missing inventory of previous 20 days)

bc_flexpart_gfas = xr.open_dataset(
    data_path + "level3/flexpart/weighted_bc_GFAS_TOTALS_ts_values_2975m_2020-2023.nc"
) * 1e9  # convert from kg/m³ to µg/m³ #new update with corrected january values (including inventory of previous 20 days)

# rename time for GFAS
bc_flexpart_gfas = bc_flexpart_gfas.rename({"release_time": "time"})
# GFAS only on 1 level (2975). For consistency, add the release_height coordinate: 
bc_flexpart_gfas = bc_flexpart_gfas.expand_dims({'release_height':bc_flexpart_gfas_old.coords["release_height"]})

# --- EDGAR ----#

bc_flexpart_edgar = xr.open_dataset(
    data_path + "level3/flexpart/weighted_bc_EDGAR_ALL_ts_values_2020_2024.nc"
) * 1e9  # convert from kg/m³ to µg/m³

# --- FULL DOMAIN ---#
# Same but for full domain
bc_flexpart_gfas_full_domain = xr.open_dataset(
    data_path + "level3/flexpart/weighted_ts_full_domain/weighted_bc_GFAS_ALL_ts_values_2975m_2020-2023.nc"
) * 1e9  # convert from kg/m³ to µg/m³
bc_flexpart_edgar_full_domain = xr.open_dataset(
    data_path + "level3/flexpart/weighted_ts_full_domain/weighted_bc_EDGAR_ALL_ts_values_2020_2024.nc"
) * 1e9  # convert from kg/m³ to µg/m³

# rename time for GFAS
bc_flexpart_gfas_full_domain = bc_flexpart_gfas_full_domain.rename({"release_time": "time"})
bc_flexpart_gfas_full_domain = bc_flexpart_gfas_full_domain.expand_dims({'release_height':bc_flexpart_gfas.coords["release_height"]})

In [ ]:
#### Read in the background BC data derived from CAMS
bc_bg1 = xr.open_dataset(
    data_path + "level3/flexpart/MKN_aermr09_BC_CAMS_EAC4_2020_2023.nc"
)['aermr09_bg'] 
bc_bg2 = xr.open_dataset(
    data_path + "level3/flexpart/MKN_aermr10_BC_CAMS_EAC4_2020_2023.nc"
)['aermr10_bg'] 
bc_bg = bc_bg1 + bc_bg2  # combine the two CAMS BC datasets
## add up hydrophilic and hydrophobic
# Note: CAMS data is in kg/kg, so I need to convert it to µg/m³


## Convert background Black carbon MMR to VMR
# determine time series of air density
# rho_d = p/(T*R) # in kg/m³
R_dry_air = 287.052874 # J/kg/K
rho_d = eac_lev * 100 / (cams_eac4_sel['t'] * R_dry_air)# in kg/m³
bc_bg = bc_bg * rho_d * 1e9 # in kg/m³ - >convert to µg/m³ with *1e9

bc_bg_daily = bc_bg.resample(time="1D").mean()

# ## Adding Anthropogenic from EDGAR and fire from GFAS
bc_all = (
    bc_bg_daily
    + bc_flexpart_gfas["bc_contribution"] 
    + bc_flexpart_edgar["bc_contribution"] 
)  
# remove agiruclture to avoid double counting:
bc_all_removed_agri = (
    bc_bg_daily
    + bc_flexpart_gfas["bc_contribution"]
    + bc_flexpart_edgar["bc_contribution"]
    - bc_flexpart_edgar["BC_agriculture"]
)

# same for full domain: 
# ## Adding Anthropogenic from EDGAR
bc_all_full_domain = (
    bc_bg_daily
    + bc_flexpart_gfas_full_domain["bc_contribution"]
    + bc_flexpart_edgar_full_domain["bc_contribution"] 
)  
# # remove agiruclture to avoid double counting:
bc_all_removed_agri_full_domain = (
    bc_bg_daily
    + bc_flexpart_gfas_full_domain["bc_contribution"]
    + bc_flexpart_edgar_full_domain["bc_contribution"]
    - bc_flexpart_edgar_full_domain["BC_agriculture"]
)

In [ ]:
# plt.figure(figsize=(figW2, figH2/2))
# aer_sel_rem_out["black_carbon_mean"].resample(time="1D").mean().plot(ls="-",label='observations')
# bc_bg.isel(release_height=0).resample(time="1D").mean().plot(c='grey',label='CAMS background')
# plt.ylabel("Black carbon (µg/m³)")
# plt.legend()
# plt.show()

# plt.figure()
# bc_flexpart_gfas_full_domain["bc_contribution"].isel(release_height=0).plot(label='full domain')
# bc_flexpart_gfas["bc_contribution"].isel(release_height=0).plot(label='nested domain')
# plt.legend()
# plt.show()

In [ ]:
## Check background

# compare CH4 background from Stephan (CAMS+flexpart) with measurements from Jungfraujoch

from processing import wdc

#data_reader_jujo =  create_data_reader(data_path="/project/leob/GAW/Kenya/data/ch4_jfj6036_surface-insitu_23_9999-9999_hourly.txt",dataset="CH4",**processing_kwargs) #creates an instance of the desired data_reader class
df_jujo = wdc.compile_wdcgg_into_dataframe("/project/leob/GAW/Kenya/data/",sampling='hourly',file_name="ch4_jfj6036_surface-insitu_23_9999-9999_hourly.txt")


In [ ]:
ds_jujo = df_jujo.to_xarray().rename({"starttime":"time"})


In [ ]:
plt.figure(figsize=(figW2, figH2/2))
ds_jujo['value'].resample(time="1D").mean().sel(time=slice('2020-01-01','2023-12-31')).plot(label='Jungfraujoch measurements')
ch4_bg_daily.isel(release_height=0).plot(label='CH4 CAMS background')
plt.legend()
plt.show()

### Plot CO, BC and CH4 contributions  <a class="anchor" id="fig_emission_contr"></a>

In [ ]:
# define release height
z_sel = 2975  # 2975m or 3678m
mw_roll = 15  # days
frac_mw_min = 0.5  # frac_mw_min Fraction of mw that is accepted as min number in a moving window (e.g. =0.5, for mw=10days it would accept 5days)
plot_rolling = True



##---- Plotting definitions ----##
figW_temp = figW3
figH_temp = 1.5*figH2
time3 = '2023-12-31' # limit those figures to year 2023

plot_species = ["CO", "BC", "CH4"] #["CO", "CH4"]

## only african domain, or full and african domain: 
plot_domains = ['','_full_domain'] # plot African domain totals and full domain totals
domain_cols = [col_contr.red,'k'] # colors for total contribution line
domain_labels = [f"African emission contribution", "Total emission contribution"]
# only african domain:
#plot_domains = ['']
#domain_cols = ['k']
# domain_labels = [f"Emission contribution (natural+anthrop.)"]

plot_anthro = True # if False, only natural sources + background are plotted

fig, axs = plt.subplots(len(plot_species), 1, figsize=(figW_temp,figH_temp), sharex=True, layout="constrained")

for i, (ax, species) in enumerate(zip(axs, plot_species)):
    ##-- define data that I want to plot
    if species == "BC":
        data = aer_sel_rem_out["black_carbon_mean"]
    else:
        data = ds_all.sel(dataset=species)["value"]

    obs = (
        data.sel(time=slice(time1, time3))
        .resample(time="1D")
        .mean()
    )

    # read in either the CO or CH4 variables (ch4_ ... or co_ ...)
    # background:
    background_to_plot = locals()[f"{species.lower()}_bg_daily"].sel(
        release_height=z_sel
    )

    # anthropogenic:

    for domain,total_col,total_label in zip(plot_domains,domain_cols,domain_labels):
        # plot first the african domain only, then the full domain

        if species == "CO" or species == "BC":
            # remove agriculture from anthropogenic contribution
            my_var = locals()[f"{species.lower()}_flexpart_edgar{domain}"]
            anthro_to_plot = background_to_plot + (
                my_var[
                    f"{species.lower()}_contribution"
                ]
                - my_var[f"{species}_agriculture"]
            )
            anthro_to_plot = anthro_to_plot.sel(release_height=z_sel)
        else:
            my_var = locals()[f"{species.lower()}_flexpart_edgar{domain}"]
            anthro_to_plot = background_to_plot + my_var[
                f"{species.lower()}_contribution"
            ].sel(release_height=z_sel)

        # natural sources and total:
        if species == "CO":
            # natural only (fires):
            my_var = locals()[f"co_flexpart_gfas{domain}"]
            natural_to_plot = (
                my_var["co_contribution"].sel(release_height=z_sel)
                + background_to_plot
            )
            # total (natural + anthropogenic):
            my_var_total = locals()[f"co_all_removed_agri{domain}"]
            total_to_plot = my_var_total.sel(
                release_height=z_sel
            )  ## removed agriculture!
        elif species == "CH4":
            my_var_wetcharts = locals()[f"ch4_flexpart_wetcharts{domain}"]
            my_var_gfas = locals()[f"ch4_flexpart_gfas{domain}"]
            natural_to_plot = (
                my_var_gfas["ch4_contribution"] 
                + my_var_wetcharts["ch4_contribution"]
            ).sel(release_height=z_sel) + background_to_plot

            total_to_plot = locals()[f"ch4_all{domain}"].sel(release_height=z_sel)  
        elif species == "BC":
            my_var_gfas = locals()[f"bc_flexpart_gfas{domain}"]
            natural_to_plot = (
                my_var_gfas["bc_contribution"] 
            ).sel(release_height=z_sel) + background_to_plot
            my_var_total = locals()[f"bc_all_removed_agri{domain}"]
            total_to_plot = my_var_total.sel(release_height=z_sel) 

        if plot_anthro == False: 
            total_to_plot = natural_to_plot #plot only natural sources + background

    ##----- Plotting        
        # natural + anthropogenic:
        #total_label = f"Total (natural+anthrop.)"
        #total_label = f"Emission contribution (natural+anthrop.)"
        #total_col = "k"

        # for now, only plot the total contribution (natural + anthropogenic)
        total_to_plot.plot(ax=ax, lw=lw_thin, alpha=marker_transp, color=total_col,)
        #plot natural separately:
        #natural_to_plot.plot(ax=ax, lw=lw_thin, alpha=marker_transp, label="Natural", color=col_bright.green if species == "CH4" else col_bright.red)

        
        #### Same but with  rolling mean
        if plot_rolling:
            # natural + anthropogenic + bckgr:
            total_to_plot.rolling(
                time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
            ).mean().plot(
                ax=ax, lw=lw_thick, alpha=marker_transp, label=total_label, color=total_col
            )

    # observations
    meas_col = col_bright.blue
    meas_label = "Measured" #f"Measured {species}"
    obs.plot(ax=ax, color=meas_col, lw=lw_thin, alpha=marker_transp, zorder=5)

    # background:
    bckgr_col = col_bright.grey
    bckgr_label = "background" #f"{species} background"
    background_to_plot.plot(
        ax=ax,
        color=bckgr_col,
        lw=lw_thin,
        alpha=marker_transp,
    )

    # # natural sources:
    # nat_label = f"Natural {species} (fires{'+wetland' if species=='CH4' else ''})"
    # nat_col = col_bright.green if species == "CH4" else col_bright.red
    # natural_to_plot.plot(ax=ax, color=nat_col, lw=lw_thin, alpha=marker_transp)

    # # anthropogenic:
    # anthro_col = col_bright.purple
    # anthro_to_plot.plot(ax=ax, color=anthro_col, lw=lw_thin, alpha=marker_transp)


    #### Same but with  rolling mean
    if plot_rolling:
        obs.rolling(    
            time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
        ).mean().plot(
            ax=ax,
            color=meas_col,
            lw=lw_thick,
            alpha=marker_transp,
            zorder=5,
            label=meas_label,
        )

        background_to_plot.rolling(
            time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
        ).mean().plot(
            ax=ax,
            color=bckgr_col,
            lw=lw_thick,
            alpha=marker_transp,
            label = bckgr_label
        )

        # natural sources:
        # natural_to_plot.rolling(
        #     time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
        # ).mean().plot(
        #     ax=ax, color=nat_col, lw=lw_thick, alpha=marker_transp, label=nat_label
        # )

        # # anthropogenic:
        # anthro_to_plot.rolling(
        # time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
        # ).mean().plot(ax=ax, color=anthro_col, lw=lw_thick, alpha=marker_transp, label="Anthropogenic")

        

    ##--- figure properties---##
    ax.set_title("")
    #ax.set_title(f"({alphabets[i]}) {species}", loc='left')

    #if species == "BC":
    ax.legend(ncols=4, loc='upper right')  #legend in every subplot
    species_str = re.sub(r"(\d+)", r"$_\1$", species)
    unit = (
        "$\mu g/$m$^3$"
        if species == "BC"
        else "ppb"
    )
    label = f"{species_str} ({unit})"
    
    # alphabet labels
    ax.text(
        0.01,
        1,
        (f"({alphabets[i]}) {species_str}"),
        transform=ax.transAxes,
        bbox=dict(fc="white", ec="none", alpha=0),
        zorder=2,
        fontsize="small",
        verticalalignment="top"
    )

    ax.set_ylabel(label)
    # ax.set_ylabel(f"{species_str} ({ds_all.sel(dataset=species).unit.values})")

    ax.set_xlabel('')
    if ax == axs[-1]:
        ax.set_xlabel("Time")                
        #ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10])) 
        #ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.xaxis.set_major_formatter(custom_month_tick_formatter)
        ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
        ax.set_xlim(pd.to_datetime(time1), pd.to_datetime(time3)) # set xlim to end of 2023

axs[0].set_title(
    f"Measurements at MKN and simulated FLEXPART-derived contribution (at {z_sel}m), {mw_roll} days rolling means",loc='left') # \n model-level: {z_sel}m"
fig.align_ylabels()

if save_fig:
    file_name_append = ''
    file_name_append = file_name_append + '_onlyNatural' if plot_anthro==False else file_name_append
    
    file_name_fig = f"{dir_save}contribution_ts_{z_sel}{file_name_append}{domain}.{fig_format}"
    ## correct figure size when saving
    save_fig_extact_size(
        fig,
        file_name_fig,
        figsize_cm=(figW_temp, figH_temp),  # Width & height in cm
        margins_cm=(0.5, 0.5, 0.5, 0.5),  # (left, bottom, right, top) in cm
        fig_dpi=fig_dpi,
        fig_format=fig_format,
    )
plt.show()

In [ ]:
# ### Same figure but only night values
# # night-time: 21UTC-4UTC (according to Henne 2008)

# # ACHTUNG: so far only for CH4 possible, as others were not folded yet on 3h basis)
# # Also, GFAS Fire is not included in methane yet!

# ##---- Plotting definitions ----##
# figW_temp = figW3
# figH_temp = 1.5*figH2
# time3 = '2023-12-31' # limit those figures to year 2023

# plot_species = ["CH4"] #["CO", "CH4"]

# fig, axs = plt.subplots(len(plot_species), 1, figsize=(figW_temp,figH_temp), sharex=True, layout="constrained")

# #for i, (ax, species) in enumerate(zip(axs, plot_species)): # not working if only one species
# # if only one species: do not loop!
# species = plot_species[0]
# i=0
# ax = axs


# ##-- define data that I want to plot
# if species == "BC":
#     data = aer_sel_rem_out["black_carbon_mean"]
# else:
#     data = ds_all.sel(dataset=species)["value"]

# obs = (
#     data.sel(time=slice(time1, time3)).where(data.time.dt.hour.isin([21,22,23,0,1,2,3,4]))
#     .resample(time="1D")
#     .mean()
# )

# # read in either the CO or CH4 variables (ch4_ ... or co_ ...)
# # background:
# background_to_plot = locals()[f"{species.lower()}_bg"].sel(
#     release_height=z_sel
# ).where(locals()[f"{species.lower()}_bg"].time.dt.hour.isin([21,22,23,0,1,2,3,4])).resample(time="1D").mean()

# # anthropogenic:

# if species == "CO" or species == "BC":
#     # remove agriculture from anthropogenic contribution
#     anthro_to_plot = background_to_plot + (
#         locals()[f"{species.lower()}_flexpart_edgar_3h"][
#             f"{species.lower()}_contribution"
#         ]
#         - locals()[f"{species.lower()}_flexpart_edgar_3"][f"{species}_agriculture"]
#     )
#     anthro_to_plot = anthro_to_plot.sel(release_height=z_sel).where(data.time.dt.hour.isin([21,22,23,0,1,2,3,4]))
# else:
#     anthro_to_plot = background_to_plot + locals()[f"{species.lower()}_flexpart_edgar_3h"][
#         f"{species.lower()}_contribution"
#     ].sel(release_height=z_sel).where(locals()[f"{species.lower()}_flexpart_edgar_3h"].time.dt.hour.isin([21,22,23,0,1,2,3,4]))

# # natural sources and total:
# if species == "CO":
#     natural_to_plot = (
#         co_flexpart_gfas_3h["co_contribution"].sel(release_height=z_sel).where(co_flexpart_gfas_3h.time.dt.hour.isin([21,22,23,0,1,2,3,4]))
#         + background_to_plot
#     )

#     total_to_plot = co_all_removed_agri_3h.sel(
#         release_height=z_sel
#     ).where(data.time.dt.hour.isin([21,22,23,0,1,2,3,4]))  ## removed agriculture!
# elif species == "CH4":
#     natural_to_plot = (
#         #ch4_flexpart_gfas["ch4_contribution"]
#         ch4_flexpart_wetcharts_3h["ch4_contribution"]
#     ).sel(release_height=z_sel).where(ch4_flexpart_wetcharts_3h.time.dt.hour.isin([21,22,23,0,1,2,3,4])) + background_to_plot

#     total_to_plot = ch4_all_3h.sel(release_height=z_sel).where(data.time.dt.hour.isin([21,22,23,0,1,2,3,4])).resample(time="1D").mean()
# elif species == "BC":
#     natural_to_plot = (
#         bc_flexpart_gfas["bc_contribution"]
#     ).sel(release_height=z_sel).where(data.time.dt.hour.isin([21,22,23,0,1,2,3,4])) + background_to_plot

#     total_to_plot = bc_all_removed_agri.sel(release_height=z_sel).where(data.time.dt.hour.isin([21,22,23,0,1,2,3,4]))

# ##----- Plotting
# # observations
# meas_col = col_bright.blue
# meas_label = "Measured" #f"Measured {species}"
# obs.plot(ax=ax, color=meas_col, lw=lw_thin, alpha=marker_transp, zorder=5)

# # background:
# bckgr_col = col_bright.grey
# bckgr_label = "background" #f"{species} background"
# background_to_plot.plot(
#     ax=ax,
#     color=bckgr_col,
#     lw=lw_thin,
#     alpha=marker_transp,
# )

# # # natural sources:
# # nat_label = f"Natural {species} (fires{'+wetland' if species=='CH4' else ''})"
# # nat_col = col_bright.green if species == "CH4" else col_bright.red
# # natural_to_plot.plot(ax=ax, color=nat_col, lw=lw_thin, alpha=marker_transp)

# # # anthropogenic:
# # anthro_col = col_bright.purple
# # anthro_to_plot.plot(ax=ax, color=anthro_col, lw=lw_thin, alpha=marker_transp)

# # natural + anthropogenic:
# #total_label = f"Total (natural+anthrop.)"
# total_label = f"Emission contribution (natural+anthrop.)"
# total_col = "k"
# total_to_plot.plot(ax=ax, color=total_col, lw=lw_thin, alpha=marker_transp)

# #### Same but with  rolling mean
# if plot_rolling:
#     obs.rolling(    
#         time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
#     ).mean().plot(
#         ax=ax,
#         color=meas_col,
#         lw=lw_thick,
#         alpha=marker_transp,
#         zorder=5,
#         label=meas_label,
#     )

#     background_to_plot.rolling(
#         time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
#     ).mean().plot(
#         ax=ax,
#         color=bckgr_col,
#         lw=lw_thick,
#         alpha=marker_transp,
#         label = bckgr_label
#     )

#     # natural sources:
#     # natural_to_plot.rolling(
#     #     time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
#     # ).mean().plot(
#     #     ax=ax, color=nat_col, lw=lw_thick, alpha=marker_transp, label=nat_label
#     # )

#     # # anthropogenic:
#     # anthro_to_plot.rolling(
#     # time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
#     # ).mean().plot(ax=ax, color=anthro_col, lw=lw_thick, alpha=marker_transp, label="Anthropogenic")

#     # natural + anthropogenic + bckgr:
#     total_to_plot.rolling(
#         time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
#     ).mean().plot(
#         ax=ax, color=total_col, lw=lw_thick, alpha=marker_transp, label=total_label
#     )

# ##--- figure properties---##
# ax.set_title("")
# #ax.set_title(f"({alphabets[i]}) {species}", loc='left')

# if species == "BC":
#     ax.legend(ncols=3, loc='upper right')  
# species_str = re.sub(r"(\d+)", r"$_\1$", species)
# unit = (
#     "$\mu g/$m$^3$"
#     if species == "BC"
#     else "ppb"
# )
# label = f"{species_str} ({unit})"

# # alphabet labels
# ax.text(
#     0.01,
#     1,
#     (f"({alphabets[i]}) {species_str}"),
#     transform=ax.transAxes,
#     bbox=dict(fc="white", ec="none", alpha=0),
#     zorder=2,
#     fontsize="small",
#     verticalalignment="top"
# )

# ax.set_ylabel(label)
# # ax.set_ylabel(f"{species_str} ({ds_all.sel(dataset=species).unit.values})")

# ax.set_xlabel('')
# #if ax == axs[-1]:
# ax.set_xlabel("Time")                
# #ax.xaxis.set_major_locator(mdates.YearLocator())
# ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10])) 
# #ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
# ax.xaxis.set_major_formatter(custom_month_tick_formatter)
# ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
# ax.set_xlim(pd.to_datetime(time1), pd.to_datetime(time3)) # set xlim to end of 2023

# #axs[0].set_title(
# ax.set_title(
#     f"Measurements at MKN and simulated FLEXPART-derived contribution at Night (at {z_sel}m), {mw_roll} days rolling means",loc='left') # \n model-level: {z_sel}m"
# fig.align_ylabels()

# if save_fig:
#     file_name_fig = f"{dir_save}contribution_ts_3h_night_{z_sel}.{fig_format}"
#     ## correct figure size when saving
#     save_fig_extact_size(
#         fig,
#         file_name_fig,
#         figsize_cm=(figW_temp, figH_temp),  # Width & height in cm
#         margins_cm=(0.5, 0.5, 0.5, 0.5),  # (left, bottom, right, top) in cm
#         fig_dpi=fig_dpi,
#         fig_format=fig_format,
#     )
# plt.show()

### Scatterplot: measurements vs. modelled  <a class="anchor" id="fig_emission_scatgter"></a>

In [ ]:

## Same but with colors for months
# 
from scipy import odr, stats # orthogonal distance regression (considering uncertainties in both variables)


## check correlations between measurements and estimated emissions
plot_species = ["CO", "BC", "CH4"] #["CO", "CH4"]
figW_temp = figW2
figH_temp = figH*1.5


# decide which regression lines to plot:
plot_ort_reg = True
plot_lin_reg = False
check_residuals = False # plot all residuals of the model fit to check

plot_anthro = True # if False, only natural sources + background are plotted


## african domain, or full domain: 
domain = '_full_domain' # plot African domain totals and full domain totals
# only african domain:
#domain = ''

fig, axs = plt.subplots(1,len(plot_species), figsize=(figW_temp,figH_temp), layout="constrained")
for i, (ax, species) in enumerate(zip(axs, plot_species)):
    ##-- define data that I want to plot
    if species == "BC":
        data = aer_sel_rem_out["black_carbon_mean"]
        obs_sd = None # no stdev available
    else:
        data = ds_all.sel(dataset=species)["value"]
        data_sd = ds_all.sel(dataset=species)["value_sd"]
        obs_sd = (
            data_sd.sel(time=slice(time1, time3))
            .resample(time="1D")
            .mean()
        )

    obs = (
        data.sel(time=slice(time1, time3))
        .resample(time="1D")
        .mean()
    )
    
    # read in either the CO or CH4 variables (ch4_ ... or co_ ...)
    # background:
    background_to_plot = locals()[f"{species.lower()}_bg_daily"].sel(
        release_height=z_sel
    )

    # anthropogenic:
    if species == "CO" or species == "BC":
        # remove agriculture from anthropogenic contribution
        my_var = locals()[f"{species.lower()}_flexpart_edgar{domain}"]
        anthro_to_plot = background_to_plot + (
            my_var[
                f"{species.lower()}_contribution"
            ]
            - my_var[f"{species}_agriculture"]
        )
        anthro_to_plot = anthro_to_plot.sel(release_height=z_sel)
    else:
        my_var = locals()[f"{species.lower()}_flexpart_edgar{domain}"]
        anthro_to_plot = background_to_plot + my_var[
            f"{species.lower()}_contribution"
        ].sel(release_height=z_sel)

    # natural sources and total:
    if species == "CO":
        my_var = locals()[f"co_flexpart_gfas{domain}"]
        natural_to_plot = (
            my_var["co_contribution"].sel(release_height=z_sel)
            + background_to_plot
        )

        my_var_total = locals()[f"co_all_removed_agri{domain}"]
        total_to_plot = my_var_total.sel(
            release_height=z_sel
            )  ## removed agriculture!
    elif species == "CH4":
        my_var_wetcharts = locals()[f"ch4_flexpart_wetcharts{domain}"]
        my_var_gfas = locals()[f"ch4_flexpart_gfas{domain}"]
        natural_to_plot = (
            my_var_gfas["ch4_contribution"] 
            + my_var_wetcharts["ch4_contribution"]
        ).sel(release_height=z_sel) + background_to_plot

        total_to_plot = locals()[f"ch4_all{domain}"].sel(release_height=z_sel)   
    elif species == "BC":
        my_var_gfas = locals()[f"bc_flexpart_gfas{domain}"]
        natural_to_plot = (
            my_var_gfas["bc_contribution"] 
        ).sel(release_height=z_sel) + background_to_plot
        my_var_total = locals()[f"bc_all_removed_agri{domain}"]
        total_to_plot = my_var_total.sel(release_height=z_sel) 

    if plot_anthro == False: 
        total_to_plot = natural_to_plot #plot only natural sources + background
        
    # correlations between obs and total_to_plot: 
    # make a pandas dataframe to use it in seaborn

    x = obs
    y = total_to_plot
    
    # per month:
    months = x.time.dt.month
    if species == "BC":
        df = pd.concat([x.to_dataframe(name='obs'), y.to_dataframe(name='sim')],axis=1) #add axis=1 to align instead of stacking vertically
    else:
        df = pd.concat([x.to_dataframe(name='obs'), y.to_dataframe(name='sim'),obs_sd.to_dataframe(name='obs_sd')],axis=1) #add standard deviation
    #df.rename(columns={"value": "CO"}, inplace=True)  # rename CO column
    df.drop("release_height", axis="columns", inplace=True)  # remove unnecessary column
    df["months"] = df.index.month
    # to group by season
    month_to_season = {m: s for s, months in my_seasons.items() for m in months}
    df["season"] = df["months"].map(month_to_season)
    df.head()

    # plt.figure()
    # # with months as colors
    # sns.jointplot(df, x="obs", y='sim', hue="months", palette=colmap, legend="full", ax = ax) # jointplot cant use subplots
    #plt.title(species)
    
    # scatterplot with months as color
    pl = sns.scatterplot(
        data=df,
        x="obs", y="sim",
        #hue="months",
        #palette=colmap,    
        hue="season",
        palette=seas_cols,  
        alpha=0.6,
        ax=ax,
        legend='full',
    )
    # get current handles/labels from seaborn
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(title='')
    if ax != axs[1]:
        ax.get_legend().remove() #keep legend only in 2nd subplot
    # regression line for all data
    # df_clean = df.dropna(subset=["obs", "sim"])
    # slope, intercept, r, p, sterr = linregress(df_clean["obs"], df_clean["sim"])
    # xx = np.linspace(df_clean["obs"].min(), df_clean["obs"].max(), 100)
    # yy = slope * xx + intercept
    # ax.plot(xx, yy, color="black", lw=2, label=f"r={r:.2f}")
    # ax.text(
    #     0.05, 0.95,
    #     f"y={slope:.2f}x+{intercept:.2f}\nr={r:.2f}",
    #     transform=ax.transAxes,
    #     ha="left", va="top", fontsize='x-small', color="black"
    # )

    # 1:1-line:
    all_vals = pd.concat([df["obs"], df["sim"]]).dropna()
    if not all_vals.empty:
        vmin, vmax = all_vals.min(), all_vals.max()
        ax.plot([vmin, vmax], [vmin, vmax], ls=':', lw=1.5, c='darkgrey')
        ax.set_xlim(vmin, vmax)
        ax.set_ylim(vmin, vmax)
        ax.set_aspect("equal", adjustable="box")
    
        # to avoid that dots are cut at the border
        vmin = vmin - 0.05*(vmax - vmin)
        vmax = vmax + 0.05*(vmax - vmin)
    
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)

    
    # regression line per season
    new_labels = []
    for season, color in seas_cols.items():
        df_seas = df[df["season"] == season].dropna(subset=["obs", "sim"])
        if df_seas.empty:
            continue

        # prepare ODR input
        if plot_ort_reg:
            # define linear model for ODR
            def f(B, x):    
                '''Linear function y = m*x + b'''
                # B is a vector of the parameters.
                # x is an array of the current x values.
                # x is in the same format as the x passed to Data or RealData.
                # Return an array in the same format as y passed to Data or RealData.
                return B[0]*x + B[1]   # B[0]=slope, B[1]=intercept
                
            model = odr.Model(f)
            if species == "BC":
                data = odr.Data(df_seas["obs"].values, df_seas["sim"].values)
            else:
                # include standard deviation of obsevations (not available for BC). This significantly improves the sum of squares!
                data = odr.RealData(df_seas["obs"].values, df_seas["sim"].values,sx=df_seas["obs_sd"].values) 
            odr_inst = odr.ODR(data, model, beta0=[1., 0.])  # initial slope=1, intercept=0
            out = odr_inst.run()
            out.pprint()

            slope, intercept = out.beta
            # corrected x values after ODR adjustments
            x_corrected = out.xplus  
            # evaluate your model at these corrected x values
            fitted_xvalues = model.fcn(out.beta, x_corrected)

            # orthogonal regression line for season
            xx = np.linspace(vmin,vmax, 1000)
            yy = slope*xx + intercept
            ax.plot(xx, yy, color=color, lw=.8, label=f"{season} fit")

            # correlation coefficient
            #corr_coef = np.corrcoef(x, out.delta)[0,1]
            corr_coef, p = stats.pearsonr(df_seas["obs"].values, df_seas["sim"].values)
            r2  = corr_coef**2
            print(f"Correlation coefficient R (obs vs. sim) ({species}, {season}): {corr_coef:.2f}")
            print(f"Coefficient of determination R² (obs vs. sim) ({species}, {season}): {r2:.2f}")

            # Add r2 (sim vs. obs) and SoS of the orthogonal fit to legend
            new_labels.append(f"{season} ({r2:.2f})") # R2 in the labels

            if check_residuals:
                # some Further checks
                print("Sum of squares:", out.sum_square) # to compare the goodness of the model fit
                # compute Pearson correlation and linear fit of delta vs x
                corr = np.corrcoef(df_seas["obs"].values, out.delta)[0,1]
                print("X-values vs. x-residuals:")
                slope2, intercept2, r, p, se = stats.linregress(df_seas["obs"].values, out.delta)  # slope quantifies trend
                print("corr, slope, p:", corr, slope2, p) # => If slope (or corr) is large and p small, the dependence is real.
                ## check residuals  
                fig_temp, ax_temp = plt.subplots()
                #ax_temp.scatter(out.eps, df_seas["sim"].values)
                ax_temp.scatter(fitted_xvalues, out.eps)
                ax_temp.set_xlabel("Fitted")
                ax_temp.set_ylabel("Residuals")
                ax_temp.set_title(f"ODR fitted x vs. y-residuals ({species}, {season})")
                fig_temp.show()

                fig_temp, ax_temp = plt.subplots()
                ax_temp.scatter( df_seas["obs"].values, out.delta)
                ax_temp.set_xlabel("Observed")
                ax_temp.set_ylabel("Residuals")
                ax_temp.set_title(f"ODR x vs. x-residuals ({species}, {season})")
                fig_temp.show()

        
        # same but with linear regression
        if plot_lin_reg: 
            slope_ols, intercept_ols, r_ols, p, sterr = linregress(df_seas["obs"].values, df_seas["sim"].values)
            xx_ols = np.linspace(vmin,vmax, 1000)
            yy_ols = slope_ols * xx_ols + intercept_ols
            ax.plot(xx_ols, yy_ols, color=color, lw=.8, label=f"r={r_ols:.2f}",ls='--')
            # Put r value on the OLS line at 80% along x
            x_pos = xx_ols[int(0.8*len(xx_ols))]
            y_pos = slope_ols*x_pos + intercept_ols
            ax.text(x_pos, y_pos, f"r={r_ols:.2f}", color=color, fontsize='x-small')

            #print(f"Regression equation: y = {slope:.2f}x + {intercept:.2f}, r = {r:.2f}, p = {p:.2f}, sterr = {sterr:.2f}")

    # # make axes equal per subplot
    # all_vals = pd.concat([df["obs"], df["sim"]]).dropna()
    # if not all_vals.empty:
    #     vmin, vmax = all_vals.min(), all_vals.max()
    #     ax.set_xlim(vmin, vmax)
    #     ax.set_ylim(vmin, vmax)

    ax.set_title("")
    species_str = re.sub(r"(\d+)", r"$_\1$", species)
    unit = (
        "$\mu g/$m$^3$"
        if species == "BC"
        else "ppb"
    )
    label = f"{species_str} ({unit})"
    ax.set_xlabel(f'Measured {label}')
    ax.set_ylabel(f'Simulated {label}')
    ax.set_title(f"({alphabets[i]}) {species_str}", loc='left')
    ax.legend(handles, new_labels, fontsize='xx-small',loc='center right') #  bbox_to_anchor=(1.05, 1), loc='upper left'

    ax.set_aspect("equal", adjustable="box")
plt.suptitle(f"Measured vs. simulated (at {z_sel}m)", y=0.8, x=0.085, ha='left')

if save_fig:
    file_name_append = ''
    file_name_append = '_ort_reg' if plot_ort_reg else ''
    file_name_append = file_name_append + 'lin_reg' if plot_lin_reg else file_name_append
    file_name_append = file_name_append + '_onlyNatural' if plot_anthro==False else file_name_append
    file_name_fig = f"{dir_save}contribution_scatter_obs_vs_sim_emissions{file_name_append}{domain}.{fig_format}"
    ## correct figure size when saving
    save_fig_extact_size(
        fig,
        file_name_fig,
        figsize_cm=(figW_temp, figH_temp),  # Width & height in cm
        margins_cm=(0.5, 0.5, 0.5, 0.5),  # (left, bottom, right, top) in cm
        fig_dpi=fig_dpi,
        fig_format=fig_format,
    )
plt.show()

In [ ]:
# ## Same but for 3h-data at day or night (only ch4 so far!!)
# ## check correlations between measurements and estimated emissions
# plot_species = ["CO", "BC", "CH4"] #["CO", "CH4"]
# plot_species = ["CO", "CH4"] #["CO", "CH4"]
# figW_temp = figW2
# figH_temp = figH*1.5

# sampling = 'day'
# if sampling == 'night':
#     include_hours = [21,22,23,0,1,2,3,4] # night-time hours
# elif sampling == 'day':
#     include_hours = list(range(5, 21)) # day-time hours


# # decide which regression lines to plot:
# plot_ort_reg = True
# plot_lin_reg = False

# fig, axs = plt.subplots(1,len(plot_species), figsize=(figW_temp,figH_temp), layout="constrained")
# for i, (ax, species) in enumerate(zip(axs, plot_species)):
#     ##-- define data that I want to plot
#     if species == "BC":
#         data = aer_sel_rem_out["black_carbon_mean"]
#         data_sd = None # no stdev available
#     else:
#         data = ds_all.sel(dataset=species)["value"]
#         data_sd = ds_all.sel(dataset=species)["value_sd"]
#         obs_sd = (
#             data_sd.sel(time=slice(time1, time3)).where(data.time.dt.hour.isin(include_hours))
#             .resample(time="1D")
#             .mean()
#         )

#     obs = (
#         data.sel(time=slice(time1, time3)).where(data.time.dt.hour.isin(include_hours))
#         .resample(time="1D")
#         .mean()
#     )
    
#     # read in either the CO or CH4 variables (ch4_ ... or co_ ...)
#     # background:
#     background_to_plot = locals()[f"{species.lower()}_bg"].sel(
#         release_height=z_sel
#     ).where(locals()[f"{species.lower()}_bg"].time.dt.hour.isin(include_hours)).resample(time="1D").mean()

#     # anthropogenic:
#     if species == "CO" or species == "BC":
#         # remove agriculture from anthropogenic contribution
#         anthro_to_plot = background_to_plot + (
#             locals()[f"{species.lower()}_flexpart_edgar"][
#                 f"{species.lower()}_contribution"
#             ]
#             - locals()[f"{species.lower()}_flexpart_edgar"][f"{species}_agriculture"]
#         )
#         anthro_to_plot = anthro_to_plot.sel(release_height=z_sel)
#     else:
#         anthro_to_plot = background_to_plot + locals()[f"{species.lower()}_flexpart_edgar_3h"][
#             f"{species.lower()}_contribution"
#         ].sel(release_height=z_sel).where(locals()[f"{species.lower()}_flexpart_edgar_3h"].time.dt.hour.isin(include_hours)).resample(time="1D").mean()

#     # natural sources and total:
#     if species == "CO":
#         natural_to_plot = (
#             co_flexpart_gfas["co_contribution"].sel(release_height=z_sel)
#             + background_to_plot
#         )

#         total_to_plot = co_all_removed_agri.sel(
#             release_height=z_sel
#         )  ## removed agriculture!
#     elif species == "CH4":
#         natural_to_plot = (
#             #ch4_flexpart_gfas["ch4_contribution"]
#             ch4_flexpart_wetcharts_3h["ch4_contribution"]
#         ).sel(release_height=z_sel).where(ch4_flexpart_wetcharts_3h.time.dt.hour.isin(include_hours)).resample(time="1D").mean() + background_to_plot

#         total_to_plot = ch4_all_3h.sel(release_height=z_sel).where(ch4_all_3h.time.dt.hour.isin(include_hours)).resample(time="1D").mean()
#     elif species == "BC":
#         natural_to_plot = (
#             bc_flexpart_gfas["bc_contribution"]
#         ).sel(release_height=z_sel).where(data.time.dt.hour.isin(include_hours)) + background_to_plot

#         total_to_plot = bc_all_removed_agri.sel(release_height=z_sel)

#     # correlations between obs and total_to_plot: 
#     # make a pandas dataframe to use it in seaborn

#     x = obs
#     y = total_to_plot
    
#     # per month:
#     months = x.time.dt.month
#     if species == "BC":
#         df = pd.concat([x.to_dataframe(name='obs'), y.to_dataframe(name='sim')],axis=1) #add axis=1 to align instead of stacking vertically
#     else:
#         df = pd.concat([x.to_dataframe(name='obs'), y.to_dataframe(name='sim'),obs_sd.to_dataframe(name='obs_sd')],axis=1) #add standard deviation
# #df.rename(columns={"value": "CO"}, inplace=True)  # rename CO column
#     df.drop("release_height", axis="columns", inplace=True)  # remove unnecessary column
#     df["months"] = df.index.month
#     # to group by season
#     month_to_season = {m: s for s, months in my_seasons.items() for m in months}
#     df["season"] = df["months"].map(month_to_season)
#     df.head()

#     # plt.figure()
#     # # with months as colors
#     # sns.jointplot(df, x="obs", y='sim', hue="months", palette=colmap, legend="full", ax = ax) # jointplot cant use subplots
#     #plt.title(species)
    
#     # scatterplot with months as color
#     pl = sns.scatterplot(
#         data=df,
#         x="obs", y="sim",
#         #hue="months",
#         #palette=colmap,    
#         hue="season",
#         palette=seas_cols,  
#         alpha=0.6,
#         ax=ax,
#         legend='full'
#     )
#     ax.legend(title='')
#     if ax != axs[1]:
#         ax.get_legend().remove() #keep legend only in 2nd subplot

#     # 1:1-line:
#     all_vals = pd.concat([df["obs"], df["sim"]]).dropna()
#     if not all_vals.empty:
#         vmin, vmax = all_vals.min(), all_vals.max()
#         ax.plot([vmin, vmax], [vmin, vmax], ls=':', lw=1.5, c='darkgrey')
#         ax.set_xlim(vmin, vmax)
#         ax.set_ylim(vmin, vmax)
#         ax.set_aspect("equal", adjustable="box")
    
#     # regression line per season
#     new_labels = []
#     for season, color in seas_cols.items():
#         df_seas = df[df["season"] == season].dropna(subset=["obs", "sim"])
#         if df_seas.empty:
#             continue

#         # prepare ODR input
#         if plot_ort_reg:
#             # define linear model for ODR
#             def f(B, x):    
#                 '''Linear function y = m*x + b'''
#                 # B is a vector of the parameters.
#                 # x is an array of the current x values.
#                 # x is in the same format as the x passed to Data or RealData.
#                 # Return an array in the same format as y passed to Data or RealData.
#                 return B[0]*x + B[1]   # B[0]=slope, B[1]=intercept
                
#             model = odr.Model(f)
#             if species == "BC":
#                 data = odr.Data(df_seas["obs"].values, df_seas["sim"].values)
#             else:
#                 # include standard deviation of obsevations (not available for BC). This significantly improves the sum of squares!
#                 data = odr.RealData(df_seas["obs"].values, df_seas["sim"].values,sx=df_seas["obs_sd"].values) 
#             odr_inst = odr.ODR(data, model, beta0=[1., 0.])  # initial slope=1, intercept=0
#             out = odr_inst.run()
#             out.pprint()

#             slope, intercept = out.beta

#             # orthogonal regression line for season
#             xx = np.linspace(vmin,vmax, 1000)
#             yy = slope*xx + intercept
#             ax.plot(xx, yy, color=color, lw=.8, label=f"{season} fit")
            
#             # correlation coefficient
#             #corr_coef = np.corrcoef(x, out.delta)[0,1]
#             corr_coef, p = stats.pearsonr(df_seas["obs"].values, df_seas["sim"].values)
#             r2  = corr_coef**2
#             print(f"Correlation coefficient R (obs vs. sim) ({species}, {season}): {corr_coef:.2f}")
#             print(f"Coefficient of determination R² (obs vs. sim) ({species}, {season}): {r2:.2f}")

#             # Add r2 (sim vs. obs) and SoS of the orthogonal fit to legend
#             new_labels.append(f"{season} ({r2:.2f})") # R2 in the labels
        
#         # same but with linear regression
#         if plot_lin_reg: 
#             slope_ols, intercept_ols, r_ols, p, sterr = linregress(df_seas["obs"].values, df_seas["sim"].values)
#             xx_ols = np.linspace(vmin,vmax, 1000)
#             yy_ols = slope_ols * xx_ols + intercept_ols
#             ax.plot(xx_ols, yy_ols, color=color, lw=.8, label=f"r={r_ols:.2f}",ls='--')
#             # Put r value on the OLS line at 80% along x
#             x_pos = xx_ols[int(0.8*len(xx_ols))]
#             y_pos = slope_ols*x_pos + intercept_ols
#             ax.text(x_pos, y_pos, f"r={r_ols:.2f}", color=color, fontsize='x-small')

#             #print(f"Regression equation: y = {slope:.2f}x + {intercept:.2f}, r = {r:.2f}, p = {p:.2f}, sterr = {sterr:.2f}")

#     # # make axes equal per subplot
#     # all_vals = pd.concat([df["obs"], df["sim"]]).dropna()
#     # if not all_vals.empty:
#     #     vmin, vmax = all_vals.min(), all_vals.max()
#     #     ax.set_xlim(vmin, vmax)
#     #     ax.set_ylim(vmin, vmax)

#     ax.set_title("")
#     species_str = re.sub(r"(\d+)", r"$_\1$", species)
#     unit = (
#         "$\mu g/$m$^3$"
#         if species == "BC"
#         else "ppb"
#     )
#     label = f"{species_str} ({unit})"
#     ax.set_xlabel(f'Measured {label}')
#     ax.set_ylabel(f'Simulated {label}')
#     ax.set_title(f"({alphabets[i]}) {species_str}", loc='left')

#     # NIGHT-time analysis only for CH4 so far!!
#     if species == "CO":
#         ax.set_title("Still all data in sim (not only night-time)!!!", loc='left')

#     ax.set_aspect("equal", adjustable="box")
#     ax.legend(handles, new_labels) #  bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.suptitle(f"Measured vs. simulated (at {z_sel}m)", y=0.8, x=0.085, ha='left')

# if save_fig:
#     file_name_append = ''
#     file_name_append = '_ort_reg' if plot_ort_reg else ''
#     file_name_append = file_name_append + 'lin_reg' if plot_lin_reg else file_name_append
#     file_name_fig = f"{dir_save}contribution_at_{sampling}time_scatter_obs_vs_sim_emissions{file_name_append}.{fig_format}"
#     ## correct figure size when saving
#     save_fig_extact_size(
#         fig,
#         file_name_fig,
#         figsize_cm=(figW_temp, figH_temp),  # Width & height in cm
#         margins_cm=(0.5, 0.5, 0.5, 0.5),  # (left, bottom, right, top) in cm
#         fig_dpi=fig_dpi,
#         fig_format=fig_format,
#     )
# plt.show()

### Plot contributions by source  <a class="anchor" id="fig_emission_conribution_ts"></a>

In [ ]:
# define category colors
colors_extend = col_bright
categories_colors = {
    "industry": colors_extend.grey,
    "transportation": colors_extend.purple,
    "waste": colors_extend.cyan,
    "energy": colors_extend.yellow,
    "fire": colors_extend.red,
    "wetlands": colors_extend.green,
    "agriculture": colors_extend.blue,
}

In [ ]:
## Anthropogenic contributions: split by source

# define release height
z_sel = 2975  # 2975m or 3678m
mw_roll = 15  # days
frac_mw_min = 0.5  # frac_mw_min Fraction of mw that is accepted as min number in a moving window (e.g. =0.5, for mw=10days it would accept 5days)
plot_rolling = True

##---- Plotting definitions ----##
figW_temp = figW3
figH_temp = 1.5*figH2

plot_species = ["CO", "BC", "CH4"] #["CO", "CH4"]

fig, axs = plt.subplots(len(plot_species), 1, figsize=(figW_temp,figH_temp), sharex=True, layout="constrained")

for i, (ax, species) in enumerate(zip(axs, plot_species)):
    ##-- define data that I want to plot
    if species == "BC":
        data = aer_sel_rem_out["black_carbon_mean"]
    else:
        data = ds_all.sel(dataset=species)["value"]

    obs = (
        data.sel(time=slice(time1, time2))
        .resample(time="1D")
        .mean()
    )

    # read in either the CO or CH4 variables (ch4_ ... or co_ ...)

    # anthropogenic:
    anthro_to_plot = locals()[f"{species.lower()}_flexpart_edgar"].sel(release_height=z_sel)
    
    # natural sources:
    #nat_label = f"Natural {species} (fires{'+wetland' if species=='CH4' else ''})"
    #nat_col = col_bright.green if species == "CH4" else col_bright.red
    if species == "CO":
        fire_to_plot = co_flexpart_gfas["co_contribution"].sel(release_height=z_sel)
        nat_label = "fire"

    elif species == "CH4":
        fire_to_plot = (
            ch4_flexpart_gfas["ch4_contribution"]
            + ch4_flexpart_wetcharts["ch4_contribution"]
        ).sel(release_height=z_sel)
        nat_label = "fire+wetland"

        # separate: 
        fire_to_plot = ch4_flexpart_gfas["ch4_contribution"].sel(release_height=z_sel)
        wetland_to_plot = ch4_flexpart_wetcharts["ch4_contribution"].sel(release_height=z_sel)
   
    elif species == "BC":
        fire_to_plot = (
            bc_flexpart_gfas["bc_contribution"]
        ).sel(release_height=z_sel) 
        nat_label = "fire"


    ##----- Plotting
    # observations
    meas_col = "tab:blue"
    meas_label = f"Measured {species}"
    # obs.plot(ax=ax, color=meas_col, lw=lw_thin, alpha=marker_transp, zorder=5)

    # natural
    fire_to_plot.plot(ax=ax, lw=lw_thin, alpha=marker_transp,c=categories_colors['fire'])
    if species == "CH4":
        wetland_to_plot.plot(ax=ax, lw=lw_thin, alpha=marker_transp,c=categories_colors['wetlands'])

    # anthropogenic:
    # set contr_cumulative to zero, with same shape as anthro_to_plot
    for (varname, da), col in zip(anthro_to_plot.data_vars.items(),col_bright):
        if varname == f"{species.lower()}_contribution":
            col_sel= "k"
            label = "TOTAL"
        else:
            label = varname.split('_')[1]
            col_sel=categories_colors[label]

        da.plot(ax=ax, lw=lw_thin, alpha=marker_transp,c=col_sel)
        # fill area between the lines



    #### Same but with  rolling mean
    if plot_rolling:
        # obs.rolling(    
        #     time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
        # ).mean().plot(
        #     ax=ax,
        #     color=meas_col,
        #     lw=lw_thick,
        #     alpha=marker_transp,
        #     zorder=5,
        #     label=meas_label,
        # )

        # natural
        fire_to_plot.rolling(
                time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
                ).mean().plot(ax=ax, lw=lw_thick, alpha=marker_transp,c=categories_colors['fire'], label='fire')
        if species == "CH4":
            wetland_to_plot.rolling(
                time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
                ).mean().plot(ax=ax, lw=lw_thick, alpha=marker_transp,c=categories_colors['wetlands'], label='wetlands')

        # anthropogenic:

        for (varname, da), col in zip(anthro_to_plot.data_vars.items(),col_bright):
            # if varname == "CO_agriculture" or varname == "BC_agriculture":
            #     continue  # skip agriculture contributions, as they are already included in fire
            if varname == f"{species.lower()}_contribution":
                col_sel= "k"
                label = "TOTAL"
            else:
                label = varname.split('_')[1]
                col_sel=categories_colors[label]
            
            da = da.rolling(
                time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
                ).mean()
            da.plot(label=label,ax=ax,lw=lw_thick, alpha=marker_transp,c=col_sel)    


    ax.set_title("")
    ax.set_title(f"({alphabets[i]}) {species}", loc='left')
    ax.legend(ncols=3)
    species_str = re.sub(r"(\d+)", r"$_\1$", species)
    unit = (
        "$\mu g/$m$^3$"
        if species == "BC"
        else "ppb"
    )
    label = (
        f"BC ({unit})" if species == "BC" else f"{species_str} ({unit})"
    )
    ax.set_ylabel(label)
    #ax.set_ylabel(f"{species_str} ({ds_all.sel(dataset=species).unit.values})")

    if ax != axs[-1]:
        ax.set_xlabel("Time")                
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,3,7,10])) 
        ax.xaxis.set_major_formatter(custom_month_tick_formatter)
        ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
        ax.set_xlim(pd.to_datetime(time1), pd.to_datetime('2023-12-31')) # set xlim to end of 2023
axs[0].set_title(
    f"Anthropogenic contribution to MKN simulations (at {z_sel}m), {mw_roll} days rolling means", loc='left') # \n model-level: {z_sel}m"
fig.align_ylabels()

if save_fig:
    file_name_fig = f"{dir_save}contribution_ts_all_contributions{z_sel}.{fig_format}"
    ## correct figure size when saving
    save_fig_extact_size(
        fig,
        file_name_fig,
        figsize_cm=(figW_temp, figH_temp),  # Width & height in cm
        margins_cm=(0.5, 0.5, 0.5, 0.5),  # (left, bottom, right, top) in cm
        fig_dpi=fig_dpi,
        fig_format=fig_format,
    )
plt.show()

In [ ]:
## Anthropogenic contributions: split by source with cumulative contributions

# define release height
z_sel = 2975  # 2975m or 3678m
mw_roll = 15  # days
frac_mw_min = 0.5  # frac_mw_min Fraction of mw that is accepted as min number in a moving window (e.g. =0.5, for mw=10days it would accept 5days)
plot_rolling = True
end_date_emissions = '2023-12-31' # we have no background for 2024, so limit it to 2023
transp_emissions = 0.7 # transparency of the emissions contributions

##---- Plotting definitions ----##
figW_temp = figW3
figH_temp = 1.5*figH2

plot_species = ["CO", "BC", "CH4"] #["CO", "CH4"]

fig, axs = plt.subplots(len(plot_species), 1, figsize=(figW_temp,figH_temp), sharex=True, layout="constrained")

for i, (ax, species) in enumerate(zip(axs, plot_species)):
    ##-- define data that I want to plot
    if species == "BC":
        data = aer_sel_rem_out["black_carbon_mean"]
    else:
        data = ds_all.sel(dataset=species)["value"]

    obs = (
        data.sel(time=slice(time1, time2))
        .resample(time="1D")
        .mean()
    )

    # read in either the CO or CH4 variables (ch4_ ... or co_ ...)

    # anthropogenic:
    anthro_to_plot = locals()[f"{species.lower()}_flexpart_edgar"].sel(release_height=z_sel,time=slice(time1, end_date_emissions))
    
    # natural sources:
    #nat_label = f"Natural {species} (fires{'+wetland' if species=='CH4' else ''})"
    #nat_col = col_bright.green if species == "CH4" else col_bright.red
    if species == "CO":
        fire_to_plot = co_flexpart_gfas["co_contribution"].sel(release_height=z_sel,time=slice(time1, end_date_emissions))
        nat_label = "fire"

    elif species == "CH4":
        # fire_to_plot = (
        #     ch4_flexpart_gfas["ch4_contribution"]
        #     + ch4_flexpart_wetcharts["ch4_contribution"]
        # ).sel(release_height=z_sel,time=slice(time1, end_date_emissions))
        # nat_label = "fire+wetlands"

        # separate: 
        fire_to_plot = ch4_flexpart_gfas["ch4_contribution"].sel(release_height=z_sel,time=slice(time1, end_date_emissions))
        wetland_to_plot = ch4_flexpart_wetcharts["ch4_contribution"].sel(release_height=z_sel,time=slice(time1, end_date_emissions))
   
    elif species == "BC":
        fire_to_plot = (
            bc_flexpart_gfas["bc_contribution"]
        ).sel(release_height=z_sel) 
        nat_label = "fire"

    ##----- Plotting
    # observations
    # meas_col = "tab:blue"
    # meas_label = f"Measured {species}"
    # obs.plot(ax=ax, color=meas_col, lw=lw_thin, alpha=marker_transp, zorder=5)

    # anthropogenic:
    # set contr_cumulative to zero, with same shape as anthro_to_plot
    # contr_cumulative = np.zeros_like(anthro_to_plot[f"{species.lower()}_contribution"])
    # for (varname, da), col in zip(anthro_to_plot.data_vars.items(),col_bright):
        # if varname == f"{species.lower()}_contribution":
        #     col_sel= "k"
        #     label = "TOTAL"
        # else:
        #     col_sel=col
        #     label = varname

    #     # ax.fill_between(
    #     #     contr_cumulative,
    #     #     (contr_cumulative+da),
    #     #     color=col_sel,
    #     #     alpha=0.3,
    #     # )
    #     contr_cumulative += da
    #     contr_cumulative.plot(label=label, ax=ax, lw=lw_thin, alpha=marker_transp,c=col_sel)
    #     # fill area between the lines



    #### Same but with  rolling mean
    if plot_rolling:
        # obs.rolling(    
        #     time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
        # ).mean().plot(
        #     ax=ax,
        #     color=meas_col,
        #     lw=lw_thick,
        #     alpha=marker_transp,
        #     zorder=5,
        #     label=meas_label,
        # )

        # set cumulative contributions to zero
        contr_cumulative_roll = np.zeros_like(fire_to_plot.rolling(
            time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
            ).mean())

        # natural
        da = fire_to_plot.rolling(time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min).mean()        
        ax.fill_between(
            da.time,
            contr_cumulative_roll,
            (contr_cumulative_roll+da),
            color=categories_colors['fire'],
            alpha=transp_emissions,
            edgecolor=None,
            label="uncontrolled fire"
        )
        contr_cumulative_roll += da  

        # for methane, plot wetland in addition (not working yet because of shorter time series)
        if species == "CH4":
            da = wetland_to_plot.rolling(time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min).mean()        
            ax.fill_between(
                da.time,
                contr_cumulative_roll,
                (contr_cumulative_roll+da),
                color=categories_colors['wetlands'],
                alpha=transp_emissions,
                edgecolor=None,
                label="wetland"
            )
            contr_cumulative_roll += da   

        # anthropogenic:
        for (varname, da) in anthro_to_plot.data_vars.items():
            if varname == "CO_agriculture" or varname == "BC_agriculture":
                continue  # skip agriculture contributions, as they are already included in fire!
            if varname == f"{species.lower()}_contribution":
                continue

            da = da.rolling(
                time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
                ).mean()
            var_label = varname.split('_')[1]
            ax.fill_between(
                da.time,
                contr_cumulative_roll,
                (contr_cumulative_roll+da),
                color=categories_colors[var_label],
                alpha=transp_emissions,
                edgecolor=None,
                label=var_label
            )
            contr_cumulative_roll += da
            #contr_cumulative_roll.plot(label=varname,ax=ax,lw=lw_thin,c=col) # additional line
        
        # # to verify if it is the same as in the previous figure: 
        # # #----
        # if species == "CO" or species == "BC":
        #     # remove agriculture from anthropogenic contribution
        #     anthro_to_plot = (
        #         locals()[f"{species.lower()}_flexpart_edgar"][
        #             f"{species.lower()}_contribution"
        #         ]
        #         - locals()[f"{species.lower()}_flexpart_edgar"][f"{species}_agriculture"]
        #     )
        #     anthro_to_plot = anthro_to_plot.sel(release_height=z_sel)
        # else:
        #     anthro_to_plot = locals()[f"{species.lower()}_flexpart_edgar"][
        #         f"{species.lower()}_contribution"
        #     ].sel(release_height=z_sel)
        # anthro_to_plot.rolling(
        #         time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
        #         ).mean().plot(label='Total anthropogenic',ax=ax,lw=lw_thick,c=anthro_col) 
        # # #---- YES it is the same as the orange anthropogenic line in the previous figure!

    ax.set_title("")
    species_str = re.sub(r"(\d+)", r"$_\1$", species)
    unit = (
        "$\mu g/$m$^3$"
        if species == "BC"
        else "ppb"
    )
    label = (
        f"BC ({unit})" if species == "BC" else f"{species_str} ({unit})"
    )
    ax.set_ylabel(label)
    #ax.set_ylabel(f"{species_str} ({ds_all.sel(dataset=species).unit.values})")

        
    # alphabet labels
    ax.text(
        0.01,
        1,
        (f"({alphabets[i]}) {species_str}"),
        transform=ax.transAxes,
        bbox=dict(fc="white", ec="none", alpha=0),
        zorder=2,
        fontsize="small",
        verticalalignment="top"
    )
    if ax == axs[0]:
        ax.legend(ncols=5)
    if ax == axs[-1]:
        ax.set_xlabel("Time")       
        ax.legend(ncols=4)         
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10])) 
        ax.xaxis.set_major_formatter(custom_month_tick_formatter)
        ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
        ax.set_xlim(pd.to_datetime(time1), pd.to_datetime(end_date_emissions)) # set xlim to end of 2023
axs[0].set_title(
    f"Natural and anthropogenic contribution to MKN simulations (at {z_sel}m), {mw_roll} days rolling means", loc='left') # \n model-level: {z_sel}m"
fig.align_ylabels()

if save_fig:
    file_name_fig = f"{dir_save}contribution_ts_cumulated_{z_sel}.{fig_format}"
    ## correct figure size when saving
    save_fig_extact_size(
        fig,
        file_name_fig,
        figsize_cm=(figW_temp, figH_temp),  # Width & height in cm
        margins_cm=(0.5, 0.5, 0.5, 0.5),  # (left, bottom, right, top) in cm
        fig_dpi=fig_dpi,
        fig_format=fig_format,
    )
plt.show()

In [ ]:
## Same figure but with percentage contributions

## Anthropogenic contributions: split by source with cumulative contributions

# define release height
z_sel = 2975  # 2975m or 3678m
mw_roll = 15  # days
frac_mw_min = 0.5  # frac_mw_min Fraction of mw that is accepted as min number in a moving window (e.g. =0.5, for mw=10days it would accept 5days)
plot_rolling = True
transp_emissions = 0.7 # transparency of the emissions contributions

##---- Plotting definitions ----##
figW_temp = figW3
figH_temp = 1.5*figH2

plot_species = ["CO", "BC", "CH4"] #["CO", "CH4"]

fig, axs = plt.subplots(len(plot_species), 1, figsize=(figW_temp,figH_temp), sharex=True, layout="constrained")

for i, (ax, species) in enumerate(zip(axs, plot_species)):
    print(species)
    ##-- define data that I want to plot
    if species == "BC":
        data = aer_sel_rem_out["black_carbon_mean"]
    else:
        data = ds_all.sel(dataset=species)["value"]

    obs = (
        data.sel(time=slice(time1, time2))
        .resample(time="1D")
        .mean()
    )

    # read in either the CO or CH4 variables (ch4_ ... or co_ ...)

    # anthropogenic:
    anthro_to_plot = locals()[f"{species.lower()}_flexpart_edgar"].sel(release_height=z_sel,time=slice(time1, end_date_emissions))
    
    # natural sources:
    #nat_label = f"Natural {species} (fires{'+wetland' if species=='CH4' else ''})"
    #nat_col = col_bright.green if species == "CH4" else col_bright.red
    if species == "CO":
        fire_to_plot = co_flexpart_gfas["co_contribution"].sel(release_height=z_sel,time=slice(time1, end_date_emissions))
        nat_label = "fire"

    elif species == "CH4":
        # fire_to_plot = (
        #     ch4_flexpart_gfas["ch4_contribution"]
        #     + ch4_flexpart_wetcharts["ch4_contribution"]
        # ).sel(release_height=z_sel,time=slice(time1, end_date_emissions))
        # nat_label = "fire+wetlands"

        # separate: 
        fire_to_plot = ch4_flexpart_gfas["ch4_contribution"].sel(release_height=z_sel,time=slice(time1, end_date_emissions))
        wetland_to_plot = ch4_flexpart_wetcharts["ch4_contribution"].sel(release_height=z_sel,time=slice(time1, end_date_emissions))
   
    elif species == "BC":
        fire_to_plot = (
            bc_flexpart_gfas["bc_contribution"]
        ).sel(release_height=z_sel) 
        nat_label = "fire"

    # Total to plot
    total_to_plot = anthro_to_plot[f"{species.lower()}_contribution"] + fire_to_plot + wetland_to_plot if species == 'CH4' else anthro_to_plot[f"{species.lower()}_contribution"]  + fire_to_plot
    # remove agriculture contributions from total, as they are already included in fire!
    if species == 'BC' or species =='CO': 
        total_to_plot = total_to_plot - anthro_to_plot[f"{species}_agriculture"] 

    #### Plot with rolling mean
    if plot_rolling:
        # set cumulative contributions to zero
        contr_cumulative_roll = np.zeros_like(fire_to_plot.rolling(
            time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
            ).mean())

        # natural relative contribution
        da = (fire_to_plot/total_to_plot *100).rolling(time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min).mean()        
        ax.fill_between(
            da.time,
            contr_cumulative_roll,
            (contr_cumulative_roll+da),
            color=categories_colors['fire'],
            alpha=transp_emissions,
            edgecolor=None,
            label="uncontrolled fire"
        )
        contr_cumulative_roll += da  
        print(f"mean fire contribution: {(fire_to_plot.mean()/total_to_plot.mean() *100):.2f}")

        # for methane, plot wetland in addition 
        if species == "CH4":
            da = (wetland_to_plot/total_to_plot*100).rolling(time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min).mean()        
            ax.fill_between(
                da.time,
                contr_cumulative_roll,
                (contr_cumulative_roll+da),
                color=categories_colors['wetlands'],
                alpha=transp_emissions,
                edgecolor=None,
                label="wetland"
            )
            contr_cumulative_roll += da   
            print(f"mean wetland contribution: {(wetland_to_plot.mean()/total_to_plot.mean() *100).mean():.2f}")

        # anthropogenic:
        for (varname, da_sect) in anthro_to_plot.data_vars.items():
            if varname == "CO_agriculture" or varname == "BC_agriculture":
                continue  # skip agriculture contributions, as they are already included in fire!
            if varname == f"{species.lower()}_contribution":
                continue

            da = (da_sect/total_to_plot*100).rolling(
                time=mw_roll, center=True, min_periods=mw_roll * frac_mw_min
                ).mean()
            var_label = varname.split('_')[1]
            ax.fill_between(
                da.time,
                contr_cumulative_roll,
                (contr_cumulative_roll+da),
                color=categories_colors[var_label],
                alpha=transp_emissions,
                edgecolor=None,
                label=var_label
            )
            contr_cumulative_roll += da      
              
            print(f"mean {varname} contribution: {(da_sect.mean()/total_to_plot.mean() *100):.2f}")

    ax.set_title("")
    species_str = re.sub(r"(\d+)", r"$_\1$", species)
    ax.set_ylabel('')
    #ax.set_ylabel(f"{species_str} ({ds_all.sel(dataset=species).unit.values})")

        
    # alphabet labels
    ax.text(
        0.01,
        .95,
        (f"({alphabets[i]}) {species_str}"),
        transform=ax.transAxes,
        bbox=dict(fc="white", ec="none", alpha=0),
        zorder=2,
        fontsize="small",
        verticalalignment="top"
    )
    if ax == axs[0]:
        ax.legend(ncols=5)
    if ax == axs[-1]:
        ax.set_xlabel("Time")       
        ax.legend(ncols=4,loc='upper right')         
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10])) 
        ax.xaxis.set_major_formatter(custom_month_tick_formatter)
        ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
        ax.set_xlim(pd.to_datetime(time1), pd.to_datetime(end_date_emissions)) # set xlim to end of 2023
axs[0].set_title(
    f"Relative natural and anthropogenic contribution to MKN simulations (at {z_sel}m), {mw_roll} days rolling means", loc='left') # \n model-level: {z_sel}m"

fig.supylabel("Sectorial emission contributions (%)")

if save_fig:
    file_name_fig = f"{dir_save}contribution_ts_cumulated_{z_sel}_REL.{fig_format}"
    ## correct figure size when saving
    save_fig_extact_size(
        fig,
        file_name_fig,
        figsize_cm=(figW_temp, figH_temp),  # Width & height in cm
        margins_cm=(0.5, 0.5, 0.5, 0.5),  # (left, bottom, right, top) in cm
        fig_dpi=fig_dpi,
        fig_format=fig_format,
    )
plt.show()

### barplots sectorial contributions <a class="anchor" id="fig_emission_conribution_bars"></a>

In [ ]:
# Create figure: one row per species, 4 bars (seasons)
figW_temp = figW2
figH_temp = figH

end_date_emissions = '2023-12-31' # we have no background for 2024, so limit it to 2023
add_bar_labels = False # add labels inside bars (percentage values)

fig, axs = plt.subplots(1, len(plot_species), figsize=(figW_temp, figH_temp), sharex=False)

for i, (ax, species) in enumerate(zip(axs, plot_species)):
    print(f"Processing {species}")
    species_str = re.sub(r"(\d+)", r"$_\1$", str(species))

    # anthropogenic:
    anthro_to_plot = locals()[f"{species.lower()}_flexpart_edgar"].sel(
        release_height=z_sel, time=slice(time1, end_date_emissions)
    )
    
    if species == "CO":
        fire_to_plot = co_flexpart_gfas["co_contribution"].sel(
            release_height=z_sel, time=slice(time1, end_date_emissions)
        )
        wetland_to_plot = None
    elif species == "CH4":
        fire_to_plot = ch4_flexpart_gfas["ch4_contribution"].sel(
            release_height=z_sel, time=slice(time1, end_date_emissions)
        )
        wetland_to_plot = ch4_flexpart_wetcharts["ch4_contribution"].sel(
            release_height=z_sel, time=slice(time1, end_date_emissions)
        )
    elif species == "BC":
        fire_to_plot = bc_flexpart_gfas["bc_contribution"].sel(
            release_height=z_sel, time=slice(time1, end_date_emissions)
        )
        wetland_to_plot = None

    # total contributions
    total_to_plot = (
        anthro_to_plot[f"{species.lower()}_contribution"] + fire_to_plot
        if species != "CH4"
        else anthro_to_plot[f"{species.lower()}_contribution"] + fire_to_plot + wetland_to_plot
    )
    if species in ["BC", "CO"]:
        total_to_plot = total_to_plot - anthro_to_plot[f"{species}_agriculture"] # remove double accounting for agriculture

    # prepare data for seasonal means
    df_contribs = {}
    df_contribs_total = {}

    # --- fire ---
    df_contribs["fire"] = (fire_to_plot.groupby("time.month").mean() / total_to_plot.groupby("time.month").mean() * 100).to_pandas()
    df_contribs_total["fire"] = (fire_to_plot.mean()/total_to_plot.mean() *100).to_pandas()
    
    # --- wetlands (CH4 only) ---
    if species == "CH4":
        df_contribs["wetlands"] = (wetland_to_plot.groupby("time.month").mean() / total_to_plot.groupby("time.month").mean() * 100).to_pandas()
        df_contribs_total["wetlands"] = (wetland_to_plot.mean()/total_to_plot.mean() *100).to_pandas()

    # --- anthropogenic subsectors ---
    for varname, da_sect in anthro_to_plot.data_vars.items():
        if varname in ["CO_agriculture", "BC_agriculture"]:  # skip double-counting
            continue
        if varname == f"{species.lower()}_contribution":
            continue
        label = varname.split("_")[1]
        df_contribs[label] = (da_sect.groupby("time.month").mean() / total_to_plot.groupby("time.month").mean() * 100).to_pandas()
        df_contribs_total[label] = (da_sect.mean()/total_to_plot.mean() *100).to_pandas()

    # --- aggregate into seasons ---
    season_means = {s: {} for s in my_seasons.keys()}
    for source, monthly_vals in df_contribs.items():
        for s, months in my_seasons.items():
            season_means[s][source] = monthly_vals.loc[months].mean()

    # --- add annual mean (total, all months)
    annual_means = {}
    for source, monthly_vals in df_contribs_total.items():
        annual_means[source] = monthly_vals.mean()
    season_means["All"] = annual_means  # add as extra "season"
    
    # --- now plot stacked bars ---
    season_labels = list(my_seasons.keys()) + ["All"]
    bottom = np.zeros(len(season_labels))

    for source, color in categories_colors.items():
        if all(source not in season_means[s] for s in season_labels):
            continue
        values = [season_means[s].get(source, 0) for s in season_labels]
        ax.bar(season_labels, values, bottom=bottom, color=color, alpha=transp_emissions, label=source)

        # add labels
        if add_bar_labels:
            for j, (val, btm) in enumerate(zip(values, bottom)):
                if val > 2:
                    ax.text(
                        j, btm + val / 2, f"{val:.1f}%",
                        #j, btm + val / 2, f"{val:.0f}",
                        ha="center", va="center", fontsize='x-small', color="k"
                    )
        bottom += values

    #ax.set_ylabel(f"{species} (%)")
    if ax == axs[0]:
        ax.set_ylabel("Emission contribution (%)")
    ax.set_title(f"({alphabets[i]}) {species_str}", loc='left')
    #ax.set_title(f"Relative contributions per season for {species}")
    # if ax != axs[-1]:
    #     ax.set_xlabel('')
    #     ax.set_xticklabels([])
    ax.tick_params(axis='x', which='minor', bottom=False)

# Put legend outside
handles, labels = ax.get_legend_handles_labels()
# Replace the fire label
labels = ['uncontrolled fire' if lbl == 'fire' else lbl for lbl in labels]
#fig.legend(handles, labels, loc="upper right",ncol=4)
lgd = ax.legend(handles, labels, bbox_to_anchor=(1.3, 0.5), loc='center', ncol=1)

plt.show()

if save_fig:
    file_name_fig = f"{dir_save}contributions_seasonal_{z_sel}_bars{'_withText' if add_bar_labels else ''}.{fig_format}"
    ## correct figure size when saving
    save_fig_extact_size(
        fig,
        file_name_fig,
        figsize_cm=(figW_temp, figH_temp),  # Width & height in cm
        margins_cm=(0.5, 0.5, 0.5, 0.5),  # (left, bottom, right, top) in cm
        fig_dpi=fig_dpi,
        fig_format=fig_format,
        #bbox_extra_artists=(lgd,), bbox_inches='tight'
    )

In [ ]:
figW_temp = figW2
figH_temp = figH2

fig, axs = plt.subplots(
    len(plot_species), 2,
    figsize=(figW_temp,figH_temp),
    sharex=False,
    gridspec_kw={"width_ratios": [2, 1]}   # left twice as wide as right
)
# Months and seasons
months = np.arange(1, 13)
season_labels = list(my_seasons.keys())


if len(plot_species) == 1:
    axs = [axs]

for i, (ax_row, species) in enumerate(zip(axs, plot_species)):
    print(f"\n=== {species} ===")

    # -----------------------------------
    # Select data for this species
    # -----------------------------------
    anthro_to_plot = locals()[f"{species.lower()}_flexpart_edgar"].sel(
        release_height=z_sel, time=slice(time1, end_date_emissions)
    )

    if species == "CO":
        fire_to_plot = co_flexpart_gfas["co_contribution"].sel(
            release_height=z_sel, time=slice(time1, end_date_emissions)
        )
        wetland_to_plot = None
    elif species == "CH4":
        fire_to_plot = ch4_flexpart_gfas["ch4_contribution"].sel(
            release_height=z_sel, time=slice(time1, end_date_emissions)
        )
        wetland_to_plot = ch4_flexpart_wetcharts["ch4_contribution"].sel(
            release_height=z_sel, time=slice(time1, end_date_emissions)
        )
    elif species == "BC":
        fire_to_plot = bc_flexpart_gfas["bc_contribution"].sel(
            release_height=z_sel, time=slice(time1, end_date_emissions)
        )
        wetland_to_plot = None

    # total contributions
    total_to_plot = (
        anthro_to_plot[f"{species.lower()}_contribution"] + fire_to_plot
        if species != "CH4"
        else anthro_to_plot[f"{species.lower()}_contribution"]
        + fire_to_plot
        + wetland_to_plot
    )
    if species in ["BC", "CO"]:
        total_to_plot = total_to_plot - anthro_to_plot[f"{species}_agriculture"]

    # -----------------------------------
    # Build contribution dictionary
    # -----------------------------------
    df_contribs = {}

    # fire
    df_contribs["fire"] = (
        fire_to_plot.groupby("time.month").mean()
        / total_to_plot.groupby("time.month").mean() * 100
    ).to_pandas()

    # wetlands (CH4 only)
    if species == "CH4":
        df_contribs["wetlands"] = (
            wetland_to_plot.groupby("time.month").mean()
            / total_to_plot.groupby("time.month").mean() * 100
        ).to_pandas()

    # anthropogenic subsectors
    for varname, da_sect in anthro_to_plot.data_vars.items():
        if varname in ["CO_agriculture", "BC_agriculture"]:
            continue
        if varname == f"{species.lower()}_contribution":
            continue
        label = varname.split("_")[1]
        df_contribs[label] = (
            da_sect.groupby("time.month").mean()
            / total_to_plot.groupby("time.month").mean() * 100
        ).to_pandas()

    sources = list(df_contribs.keys())

    # -----------------------------------
    # 1) Monthly cycle (stacked area)
    # -----------------------------------
    contrib_matrix = np.vstack(
        [df_contribs[src].reindex(months, fill_value=0).values for src in sources]
    )

    ax_month = ax_row[0]
    ax_month.stackplot(
        months,
        contrib_matrix,
        labels=sources,
        colors=[categories_colors[src] for src in sources],
        alpha=transp_emissions,
    )
    ax_month.set_xlim(1, 12)
    ax_month.set_xticks(months)
    ax_month.set_xticklabels(
        ["Jan","Feb","Mar","Apr","May","Jun",
         "Jul","Aug","Sep","Oct","Nov","Dec"]
    )
    ax_month.set_ylabel(f"{species} (%)")
    ax_month.set_title(f"{species}: Monthly contributions")

    # -----------------------------------
    # 2) Seasonal stacked bars
    # -----------------------------------
    season_means = {s: {} for s in my_seasons.keys()}
    for source, monthly_vals in df_contribs.items():
        for s, months_seas in my_seasons.items():
            season_means[s][source] = monthly_vals.loc[months_seas].mean()

    ax_seas = ax_row[1]
    bottom = np.zeros(len(season_labels))

    for source in sources:
        values = [season_means[s].get(source, 0) for s in season_labels]
        ax_seas.bar(
            season_labels,
            values,
            bottom=bottom,
            color=categories_colors[source],
            alpha=transp_emissions,
            label=source,
        )
        # add labels inside bars
        for j, (val, btm) in enumerate(zip(values, bottom)):
            if val > 2:  # skip very small contributions
                ax_seas.text(
                    j, btm + val / 2,
                    f"{val:.1f}%",
                    ha="center", va="center",
                    fontsize=8, color="black",
                )
        bottom += values

    ax_seas.set_ylabel(f"{species} (%)")
    ax_seas.set_title(f"{species}: Seasonal contributions")

# -----------------------------------
# Legend (once for all)
# -----------------------------------
handles, labels = ax_month.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")

if save_fig:
    file_name_fig = f"{dir_save}{'_'.join(plot_species)}_seasonal_contributions_months_seas_{z_sel}_REL.{fig_format}"
    ## correct figure size when saving
    save_fig_extact_size(
        fig,
        file_name_fig,
        figsize_cm=(figW_temp, figH_temp),  # Width & height in cm
        margins_cm=(0.5, 0.5, 0.5, 0.5),  # (left, bottom, right, top) in cm
        fig_dpi=fig_dpi,
        fig_format=fig_format,
    )
